# VeritasAI — Truth-Driven News Intelligence

VeritasAI is a real-time AI-powered news aggregator agent built for the
Application Developer RA position at Rutgers MPI. This notebook contains
the complete project in a single Colab file — from data fetching all the
way to a fully custom web interface accessible from any browser.

The core idea is simple. Most news aggregators are pipelines. They follow
the same fixed steps every time regardless of what they find. VeritasAI is
built around a genuine AI agent loop using Google's flan-t5-base open-source
language model. After every action, the agent observes its results, reasons
about quality, and decides what to do next. The sequence of steps adapts to
each topic individually.

---

## How to Run

1. Run Cell 1 (this cell is just text, skip to Cell 2)
2. Run Cell 2 to install all dependencies — takes 2 to 3 minutes, run once only
3. Run all remaining cells from top to bottom in order
4. Run the final launch cell — it will print a public URL ending in trycloudflare.com
5. Open that URL in your browser and start searching

No API keys, no accounts, and no tokens are required.

---

## Features

### Core Requirements
- Three topic inputs on the frontend search page
- AI agent fetches the 10 most recent articles per topic from Google News RSS
- VADER sentiment classification per headline (Positive, Negative, Neutral)
- Articles sorted by publication date descending, most recent first
- All headlines displayed as clickable links with source and date metadata

### AI Agent (flan-t5-base)
- Google flan-t5-base open-source language model as the reasoning engine
- Six tools in the tool registry: search_news, assess_quality, refine_query,
  broaden_query, analyze_sentiment, summarize
- Observe-reason-act loop running up to 3 retry attempts per topic
- Topic specificity assessed by spaCy NER before the first search
- Quality assessed by checking article count and headline relevance
- Full reasoning log shown in the sidebar after each search

### Text Visualizations (Voyant-inspired)
- Word cloud with circular mask and custom color palette
- Sentiment distribution donut chart
- Top 15 keyword frequency bar chart
- Sentiment score scatter plot over publication time

### Multilingual Translation
- Helsinki-NLP OPUS-MT open-source models from HuggingFace
- Supports Spanish, Hindi, French, and German
- Language toggle in the frontend, models cache after first load

### VeritasBot (Optional Chat)
- Responds to natural language queries like "Find news on Intel stock price"
- Returns 1 to 3 clickable article URLs plus a courtesy comment
- Six enhancements: spaCy NER query understanding, multi-turn memory,
  sentiment-aware response tone, follow-up suggestion chips, TextRank
  briefing mode, and sentiment percentage commentary on every response

---

## Tech Stack

| Tool | Purpose |
|---|---|
| flan-t5-base (HuggingFace) | Open-source LLM for agent reasoning |
| feedparser | Google News RSS parsing, no API key needed |
| VADER (vaderSentiment) | Headline sentiment classification |
| spaCy en_core_web_sm | Named entity recognition |
| sumy TextRank | Extractive summarization |
| Helsinki-NLP OPUS-MT | Open-source neural machine translation |
| wordcloud + numpy | Word cloud with circular mask |
| Plotly | Interactive charts |
| python-dateutil | Robust date string parsing |
| FastAPI + uvicorn | Backend web framework |
| Cloudflare Tunnel | Free public URL, no account needed |

---

## AI Usage Declaration

This notebook was developed with assistance from an AI coding assistant
during development only. The AI was not integrated into the running
application at any point. All LLMs used inside the app at runtime
(flan-t5-base and Helsinki-NLP OPUS-MT) are fully open-source and
publicly available on HuggingFace.

The AI assistant helped with: translation pipeline integration, FastAPI
routing structure, and debugging library compatibility issues. All core
design decisions including the agent architecture, tool registry, chatbot
enhancement logic, and frontend layout were made independently.

---

## References

All references are listed in the final cell of this notebook.


## Step 1 — Install All Runtime Dependencies

This cell installs every Python library the project needs, downloads the
spaCy English NER model, fetches the NLTK tokenizer data required by sumy
TextRank, and downloads the cloudflared binary that creates the public URL.

Run this cell first and wait for it to finish before running any other cell.
In a fresh Colab session this takes about 2 to 3 minutes depending on
network speed.

Libraries installed:
- fastapi, uvicorn: web framework and server
- feedparser: RSS feed parsing
- vaderSentiment: headline sentiment analysis
- python-dateutil: robust date parsing
- spacy: named entity recognition
- sumy: TextRank extractive summarization
- wordcloud, matplotlib, numpy: visualization support
- plotly: interactive charts
- transformers, sentencepiece, sacremoses, accelerate: HuggingFace model loading
- nltk: tokenizer data for sumy
- torch (CPU build): required by transformers for model inference


In [ ]:

import os
import stat
import subprocess
import sys
from pathlib import Path

PACKAGES = [
    "fastapi",
    "uvicorn",
    "feedparser",
    "vaderSentiment",
    "python-dateutil",
    "spacy",
    "sumy",
    "wordcloud",
    "plotly",
    "transformers",
    "sentencepiece",
    "sacremoses",
    "accelerate",
    "nltk",
    "matplotlib",
    "numpy",
    "torch",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)

import nltk

for resource_name in ("punkt", "punkt_tab"):
    try:
        nltk.download(resource_name, quiet=True)
    except Exception:
        pass

CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
cloudflared_path = Path("/content/cloudflared")

def download_cloudflared(target: Path) -> None:
    commands = [
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "4",
            "--retry-all-errors",
            "--connect-timeout",
            "20",
            CLOUDFLARED_URL,
            "-o",
            str(target),
        ],
        [
            "wget",
            "-q",
            "--tries=4",
            "--timeout=20",
            CLOUDFLARED_URL,
            "-O",
            str(target),
        ],
    ]
    errors = []

    for command in commands:
        try:
            if target.exists():
                target.unlink()
            subprocess.run(command, check=True)
            if target.exists() and target.stat().st_size > 0:
                target.chmod(target.stat().st_mode | stat.S_IEXEC)
                return
            errors.append(f"{command[0]} produced an empty file")
        except Exception as exc:
            errors.append(f"{command[0]} failed: {exc}")

    raise RuntimeError(
        "Cloudflared download failed after retrying with curl and wget. "
        + " | ".join(errors)
    )

if not cloudflared_path.exists() or cloudflared_path.stat().st_size == 0:
    download_cloudflared(cloudflared_path)

print("Setup complete. Runtime dependencies and cloudflared are ready.")


## Step 2 — News Fetching Module

This module is the first tool in the agent's tool registry. Every time the
agent decides to fetch news it calls the functions defined here.

The fetcher uses Google News RSS because it is free, requires no API key,
returns real-time results for any search query, and includes publication
timestamps which are needed for sorting. The feedparser library parses the
RSS feed into Python objects.

Each article comes back as a dictionary with these fields:
- headline: cleaned title with source suffix removed
- translated_headline: initially same as headline, updated by the translator
- link: direct URL to the full article
- source: publisher name extracted from the RSS entry
- published: raw date string from the RSS feed
- topic: which search topic this article belongs to

The fetch_all_topics function loops over up to 3 topics and combines results
into one flat list for the agent to process.


In [ ]:
%%writefile /content/fetch_news.py
from __future__ import annotations

import html
import re
import time
from urllib.error import HTTPError, URLError
from urllib.parse import quote_plus
from urllib.request import Request, urlopen

import feedparser


GOOGLE_NEWS_TEMPLATE = (
    "https://news.google.com/rss/search?q={query}&hl={hl}&gl={gl}&ceid={ceid}"
)
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; VeritasAI/1.0; +https://news.google.com)"
}


def build_google_news_url(query: str, language: str = "en-US", country: str = "US") -> str:
    safe_query = quote_plus((query or "").strip())
    return GOOGLE_NEWS_TEMPLATE.format(
        query=safe_query,
        hl=language,
        gl=country,
        ceid=f"{country}:en",
    )


def normalize_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", (value or "")).strip()


def strip_tags(value: str) -> str:
    return normalize_whitespace(re.sub(r"<[^>]+>", " ", html.unescape(value or "")))


def download_feed(url: str, attempts: int = 2, timeout: int = 8) -> bytes:
    for attempt in range(attempts):
        try:
            request = Request(url, headers=DEFAULT_HEADERS)
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError, OSError):
            if attempt + 1 < attempts:
                time.sleep(0.6)
    return b""


def entry_get(entry, key: str, default=None):
    if hasattr(entry, "get"):
        try:
            return entry.get(key, default)
        except Exception:
            pass
    return getattr(entry, key, default)


def split_headline_source(title: str) -> tuple[str, str]:
    clean_title = normalize_whitespace(html.unescape(title))
    if " - " in clean_title:
        headline, source = clean_title.rsplit(" - ", 1)
        if 1 <= len(source.split()) <= 6:
            return headline.strip(), source.strip()
    return clean_title, "Unknown Source"


def article_from_entry(entry, topic: str, query: str) -> dict:
    raw_title = getattr(entry, "title", "") or ""
    headline, fallback_source = split_headline_source(raw_title)
    source = fallback_source
    entry_source = getattr(entry, "source", None)
    if entry_source and getattr(entry_source, "title", None):
        source = normalize_whitespace(entry_source.title)
    raw_summary = (
        entry_get(entry_get(entry, "summary_detail", {}), "value", "")
        or getattr(entry, "summary", "")
        or ""
    )

    return {
        "topic": topic,
        "query": query,
        "headline": headline,
        "translated_headline": headline,
        "link": getattr(entry, "link", "") or "",
        "source": source or "Unknown Source",
        "published": getattr(entry, "published", "")
        or getattr(entry, "updated", "")
        or "",
        "raw_title": raw_title,
        "summary": strip_tags(raw_summary),
    }


def fetch_news(
    topic: str,
    max_articles: int = 15,
    language: str = "en-US",
    country: str = "US",
    display_topic: str | None = None,
) -> list[dict]:
    topic = (topic or "").strip()
    display_topic = (display_topic or topic).strip()
    if not topic:
        return []

    url = build_google_news_url(topic, language=language, country=country)
    feed_bytes = download_feed(url)
    if not feed_bytes:
        return []

    parsed = feedparser.parse(feed_bytes)
    entries = getattr(parsed, "entries", []) or []
    articles = []

    for entry in entries[: max_articles or 15]:
        try:
            article = article_from_entry(entry, topic=display_topic, query=topic)
            if article["headline"] and article["link"]:
                articles.append(article)
        except Exception:
            continue

    return articles


def fetch_all_topics(topics: list[str], max_articles: int = 15) -> list[dict]:
    all_articles = []
    for topic in topics or []:
        all_articles.extend(fetch_news(topic, max_articles=max_articles, display_topic=topic))
    return all_articles


## Step 3 — Sentiment and Date Processing Module

This module does two things to every article. First it parses the raw date
string into a proper datetime object so articles can be sorted correctly.
Second it runs every headline through VADER sentiment analysis.

VADER stands for Valence Aware Dictionary and sEntiment Reasoner. It was
built specifically for short social media and news-style text, which makes
it more accurate on headlines than general-purpose models. It handles
negation, capitalization, and punctuation automatically.

VADER returns a compound score between -1.0 and +1.0. We map this to three
labels using the thresholds recommended in the original research paper:
- 0.05 and above: Positive
- -0.05 and below: Negative
- Between -0.05 and 0.05: Neutral

The process_articles function also removes duplicate headlines and sorts
everything by date descending so the most recent article always appears first,
satisfying the assignment requirement directly.


In [ ]:
%%writefile /content/sentiment.py
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from typing import Iterable

from dateutil import parser as date_parser
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


analyzer = SentimentIntensityAnalyzer()
SENTIMENT_LABELS = ("Positive", "Negative", "Neutral")


def parse_date(date_string: str) -> dict:
    if not date_string:
        parsed = datetime.now(timezone.utc)
    else:
        try:
            parsed = date_parser.parse(date_string)
        except Exception:
            parsed = datetime.now(timezone.utc)

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    parsed = parsed.astimezone(timezone.utc)
    return {
        "published_iso": parsed.isoformat(),
        "published_display": parsed.strftime("%b %d, %Y %H:%M UTC"),
        "timestamp": parsed.timestamp(),
    }


def get_sentiment(headline: str) -> tuple[str, float]:
    score = analyzer.polarity_scores((headline or "").strip()).get("compound", 0.0)
    if score >= 0.05:
        label = "Positive"
    elif score <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    return label, round(float(score), 3)


def _dedupe_key(article: dict) -> tuple[str, str, str]:
    return (
        (article.get("topic") or "").strip().lower(),
        (article.get("link") or "").strip().lower(),
        (article.get("headline") or "").strip().lower(),
    )


def process_articles(articles: Iterable[dict], limit_per_topic: int = 10) -> list[dict]:
    unique_articles: dict[tuple[str, str, str], dict] = {}

    for article in articles or []:
        enriched = dict(article)
        enriched.setdefault("translated_headline", enriched.get("headline", ""))
        enriched.update(parse_date(enriched.get("published", "")))
        sentiment, score = get_sentiment(enriched.get("headline", ""))
        enriched["sentiment"] = sentiment
        enriched["sentiment_score"] = score

        key = _dedupe_key(enriched)
        existing = unique_articles.get(key)
        if existing is None or enriched["timestamp"] > existing["timestamp"]:
            unique_articles[key] = enriched

    grouped: dict[str, list[dict]] = defaultdict(list)
    for article in unique_articles.values():
        grouped[article.get("topic") or "General"].append(article)

    trimmed_articles: list[dict] = []
    for group in grouped.values():
        group.sort(key=lambda item: item["timestamp"], reverse=True)
        trimmed_articles.extend(group[:limit_per_topic])

    trimmed_articles.sort(key=lambda item: item["timestamp"], reverse=True)
    return trimmed_articles


def sentiment_breakdown(articles: Iterable[dict]) -> dict:
    counts = {label: 0 for label in SENTIMENT_LABELS}
    article_list = list(articles or [])

    for article in article_list:
        label = article.get("sentiment", "Neutral")
        if label not in counts:
            label = "Neutral"
        counts[label] += 1

    total = len(article_list)
    percentages = {
        label: round((count / total) * 100, 1) if total else 0.0
        for label, count in counts.items()
    }

    return {
        "total": total,
        "counts": counts,
        "percentages": percentages,
    }


## Step 4 — NLP Utility Module

This module handles two NLP tasks that are used in different parts of
the project.

The first is Named Entity Recognition using spaCy. When a user types
something into VeritasBot like "what is happening with Apple?" the bot
needs to figure out that Apple is the topic to search. spaCy's NER model
identifies real-world entities like organizations, people, and locations
regardless of how the sentence is phrased.

The same NER logic is used by the agent to assess whether a topic is
specific enough to search directly or too vague and needs expansion first.

The second task is TextRank summarization via the sumy library. TextRank
is a graph-based algorithm that identifies the most representative sentences
in a body of text by measuring how similar each sentence is to all the
others. The most connected sentences win. This is the same technique used
by Google News Briefings and it runs entirely locally with no additional
model download needed.

TextRank powers two features: the News Brief card that appears at the top
of results after each fetch, and the briefing mode in VeritasBot when the
user asks for a summary.


In [ ]:
%%writefile /content/nlp_pipeline.py
from __future__ import annotations

from collections import Counter
from functools import lru_cache

import spacy
from sumy.nlp.tokenizers import Tokenizer
from sumy.parsers.plaintext import PlaintextParser
from sumy.summarizers.text_rank import TextRankSummarizer


ALLOWED_ENTITY_LABELS = {
    "PERSON",
    "ORG",
    "GPE",
    "LOC",
    "PRODUCT",
    "EVENT",
    "NORP",
    "WORK_OF_ART",
}

VAGUE_TOPICS = {
    "ai",
    "news",
    "technology",
    "economy",
    "markets",
    "business",
    "politics",
    "science",
    "weather",
    "sports",
    "war",
    "finance",
    "energy",
    "trade",
}


@lru_cache(maxsize=1)
def get_nlp():
    return spacy.load("en_core_web_sm")


def extract_entities_from_query(query: str) -> list[str]:
    query = (query or "").strip()
    if not query:
        return []

    doc = get_nlp()(query)
    entities = []
    seen = set()

    for ent in doc.ents:
        cleaned = ent.text.strip()
        if cleaned and cleaned.lower() not in seen:
            seen.add(cleaned.lower())
            entities.append(cleaned)

    return entities


def extract_keywords(articles: list[dict], top_n: int = 15) -> list[dict]:
    text = " ".join(article.get("headline", "") for article in articles or [])
    if not text.strip():
        return []

    doc = get_nlp()(text)
    counter = Counter()

    for token in doc:
        if token.is_stop or token.is_punct or token.is_space or token.like_num:
            continue
        if token.pos_ not in {"NOUN", "PROPN", "ADJ"}:
            continue
        cleaned = token.lemma_.strip().lower()
        if len(cleaned) < 3:
            continue
        counter[cleaned] += 1

    return [
        {"term": term, "count": count}
        for term, count in counter.most_common(top_n)
    ]


def generate_briefing(articles: list[dict], sentence_count: int = 3) -> str:
    if not articles:
        return "No live articles were available yet, so the briefing will appear after the first successful fetch."

    headline_text = ". ".join(article.get("headline", "") for article in articles if article.get("headline"))
    if headline_text.count(".") < 2:
        fallback = [article.get("headline", "") for article in articles[:3]]
        return " ".join(part for part in fallback if part).strip()

    try:
        parser = PlaintextParser.from_string(headline_text, Tokenizer("english"))
        summarizer = TextRankSummarizer()
        summary = summarizer(parser.document, sentence_count)
        rendered = " ".join(str(sentence) for sentence in summary).strip()
        if rendered:
            return rendered
    except Exception:
        pass

    fallback = [article.get("headline", "") for article in articles[:3]]
    return " ".join(part for part in fallback if part).strip()


def extract_followup_suggestions(
    articles: list[dict],
    topic: str,
    limit: int = 3,
) -> list[str]:
    combined_text = " ".join(article.get("headline", "") for article in articles or [])
    if not combined_text.strip():
        return []

    doc = get_nlp()(combined_text)
    suggestions = []
    seen = {(topic or "").strip().lower()}

    for ent in doc.ents:
        cleaned = ent.text.strip()
        lowered = cleaned.lower()
        if not cleaned or ent.label_ not in ALLOWED_ENTITY_LABELS:
            continue
        if lowered in seen or len(cleaned) < 3:
            continue
        suggestions.append(cleaned)
        seen.add(lowered)
        if len(suggestions) >= limit:
            return suggestions

    for keyword in extract_keywords(articles, top_n=10):
        term = keyword["term"].replace("_", " ").title()
        lowered = term.lower()
        if lowered in seen:
            continue
        suggestions.append(term)
        seen.add(lowered)
        if len(suggestions) >= limit:
            break

    return suggestions[:limit]


def assess_topic_specificity(topic: str) -> str:
    topic = (topic or "").strip()
    if not topic:
        return "vague"

    doc = get_nlp()(topic)
    entity_labels = {ent.label_ for ent in doc.ents}
    lowered = topic.lower()

    if "PERSON" in entity_labels:
        return "person"
    if entity_labels & {"ORG", "PRODUCT", "GPE", "LOC", "EVENT", "NORP"}:
        return "specific"
    if len(topic.split()) >= 2:
        return "specific"
    if lowered in VAGUE_TOPICS:
        return "vague"

    token_count = sum(1 for token in doc if not token.is_space and not token.is_punct)
    if token_count <= 1:
        return "vague"
    return "specific"


## Step 5 — Multilingual Translation Module

This module adds multilingual support to VeritasAI. When the user selects
a language from the dropdown, all fetched headlines translate instantly to
that language without re-fetching the news.

Translation uses Helsinki-NLP OPUS-MT models from HuggingFace. These are
open-source neural machine translation models trained on millions of
sentence pairs. There is a separate model for each language direction —
English to Spanish, English to Hindi, English to French, and English to
German. Each model is around 300MB.

To avoid loading all models at startup, each model loads only when the user
selects that language for the first time. After that it is cached using
Python's lru_cache so subsequent translations for the same language are
near-instant. The first translation for a new language takes 20 to 40
seconds while the model downloads.

All models run locally inside Colab. No data is sent to any external API
and no authentication is required.


In [ ]:
%%writefile /content/translator.py
from __future__ import annotations

from functools import lru_cache

from transformers import MarianMTModel, MarianTokenizer


LANGUAGE_CONFIG = {
    "English": None,
    "Spanish": "Helsinki-NLP/opus-mt-en-es",
    "Hindi": "Helsinki-NLP/opus-mt-en-hi",
    "French": "Helsinki-NLP/opus-mt-en-fr",
    "German": "Helsinki-NLP/opus-mt-en-de",
}


@lru_cache(maxsize=4)
def load_translator(language: str):
    model_name = LANGUAGE_CONFIG.get(language)
    if not model_name:
        return None, None

    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model


def translate_headline(headline: str, language: str) -> str:
    if language not in LANGUAGE_CONFIG or language == "English":
        return headline

    try:
        tokenizer, model = load_translator(language)
        if tokenizer is None or model is None:
            return headline

        batch = tokenizer(
            [headline],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        )
        generated = model.generate(**batch, max_length=128)
        return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    except Exception:
        return headline


def translate_articles(articles: list[dict], language: str) -> list[dict]:
    translated = []
    for article in articles or []:
        enriched = dict(article)
        enriched["translated_headline"] = translate_headline(
            article.get("headline", ""),
            language,
        )
        translated.append(enriched)
    return translated


## Step 6 — Visualization Module

This module generates four visual elements that appear in the Insights
section of the frontend. Each visualization answers a specific question
about the fetched article set.

Word Cloud: What words dominate the headlines right now? Combines all
headlines, filters stopwords, and renders a circular word cloud where
larger words appear more frequently. Uses a custom color function with
near-black, grey, and gold shades. The circular shape uses a numpy mask.

Sentiment Pie Chart: Is overall coverage positive, negative, or neutral?
An interactive donut chart showing proportions across all fetched articles.

Keyword Bar Chart: Which specific terms appear most often? The top 15
most frequent non-stopword terms ranked by count.

Sentiment Scatter Plot: Has coverage sentiment shifted over time? Each
article is plotted as a dot with publication time on the x-axis and VADER
compound score on the y-axis. Dashed threshold lines at +0.05 and -0.05
show the positive and negative boundaries.

All charts are built with Plotly and sent to the frontend as JSON. The
browser renders them using the Plotly CDN with full interactivity.


In [ ]:
%%writefile /content/visualizations.py
from __future__ import annotations

import base64
import io
import json
import random
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from wordcloud import WordCloud

from nlp_pipeline import extract_keywords
from sentiment import sentiment_breakdown


WORDCLOUD_COLORS = ["#2D8A70", "#1A1A1A", "#708090"]


def _wordcloud_color_func(*_args, **_kwargs):
    return random.choice(WORDCLOUD_COLORS)


def _build_circle_mask(size: int = 700) -> np.ndarray:
    y, x = np.ogrid[:size, :size]
    center = (size - 1) / 2
    radius = size / 2.25
    mask = np.where((x - center) ** 2 + (y - center) ** 2 > radius ** 2, 255, 0)
    return mask.astype(np.uint8)


def generate_wordcloud(articles: list[dict]) -> str | None:
    text = " ".join(article.get("headline", "") for article in articles or [])
    if not text.strip():
        return None

    mask = _build_circle_mask()
    cloud = WordCloud(
        width=900,
        height=900,
        background_color=None,
        mode="RGBA",
        mask=mask,
        contour_width=0,
        max_words=180,
        collocations=False,
    ).generate(text)
    cloud.recolor(color_func=_wordcloud_color_func)

    fig, ax = plt.subplots(figsize=(7, 7), facecolor="none")
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")
    ax.imshow(cloud, interpolation="bilinear")
    ax.axis("off")

    buffer = io.BytesIO()
    plt.tight_layout(pad=0)
    plt.savefig(buffer, format="png", bbox_inches="tight", transparent=True)
    plt.close(fig)
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


def generate_sentiment_pie(articles: list[dict]) -> dict:
    breakdown = sentiment_breakdown(articles)
    counts = breakdown["counts"]

    fig = go.Figure(
        data=[
            go.Pie(
                labels=list(counts.keys()),
                values=list(counts.values()),
                hole=0.62,
                marker=dict(colors=["#5aa469", "#d65d5d", "#b0a99f"]),
                textinfo="label+percent",
            )
        ]
    )
    fig.update_layout(
        title="Sentiment Distribution",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        font=dict(family="Inter, sans-serif"),
    )
    return json.loads(fig.to_json())


def generate_keyword_bar(articles: list[dict]) -> dict:
    keywords = extract_keywords(articles, top_n=15)
    terms = [item["term"].title() for item in keywords]
    counts = [item["count"] for item in keywords]

    fig = go.Figure(
        data=[
            go.Bar(
                x=counts,
                y=terms,
                orientation="h",
                marker=dict(
                    color=counts,
                    colorscale=[
                        [0.0, "#7b5b20"],
                        [0.5, "#b8892d"],
                        [1.0, "#e1bc5c"],
                    ],
                ),
            )
        ]
    )
    fig.update_layout(
        title="Top Keywords",
        yaxis=dict(autorange="reversed"),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        font=dict(family="Inter, sans-serif"),
    )
    return json.loads(fig.to_json())


def generate_sentiment_scatter(articles: list[dict]) -> dict:
    ordered = sorted(articles or [], key=lambda item: item.get("timestamp", 0))
    x_values = []
    y_values = []
    colors = []
    labels = []

    for article in ordered:
        timestamp = article.get("timestamp")
        if timestamp:
            dt = datetime.fromtimestamp(timestamp, tz=timezone.utc)
            x_values.append(dt.isoformat())
        else:
            x_values.append(article.get("published_iso", ""))

        score = article.get("sentiment_score", 0.0)
        y_values.append(score)
        labels.append(article.get("headline", ""))

        sentiment = article.get("sentiment", "Neutral")
        if sentiment == "Positive":
            colors.append("#4f9d69")
        elif sentiment == "Negative":
            colors.append("#d4685d")
        else:
            colors.append("#8b847e")

    fig = go.Figure(
        data=[
            go.Scatter(
                x=x_values,
                y=y_values,
                mode="markers",
                marker=dict(size=11, color=colors, line=dict(width=1, color="#ffffff")),
                text=labels,
                hovertemplate="%{text}<br>Score: %{y}<extra></extra>",
            )
        ]
    )
    fig.add_hline(y=0.05, line_dash="dash", line_color="#5aa469")
    fig.add_hline(y=-0.05, line_dash="dash", line_color="#d65d5d")
    fig.update_layout(
        title="Sentiment Score Over Time",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        xaxis_title="Publication Time",
        yaxis_title="VADER Compound Score",
        font=dict(family="Inter, sans-serif"),
    )
    return json.loads(fig.to_json())


def build_visual_payload(articles: list[dict]) -> dict:
    return {
        "wordcloud": generate_wordcloud(articles),
        "pie": generate_sentiment_pie(articles),
        "bar": generate_keyword_bar(articles),
        "scatter": generate_sentiment_scatter(articles),
    }


## Step 7 — VeritasAgent (The AI Brain)

This is the module that makes VeritasAI genuinely different from a standard
news aggregator pipeline. A pipeline always follows the same steps in the
same order. VeritasAgent does not.

The agent uses Google's flan-t5-base open-source language model as its
reasoning engine. flan-t5-base is an instruction-tuned sequence-to-sequence
model trained to follow natural language prompts. It runs on CPU inside
Colab without requiring a GPU.

The observe-reason-act loop works like this for each topic:

1. OBSERVE: The agent reads the topic and uses spaCy to assess how
   specific it is (specific, vague, or person entity)

2. REASON: flan-t5 decides whether to search directly or expand the
   query first. Specific topics like company names search directly.
   Vague topics get expanded.

3. ACT: The agent calls the search tool and fetches articles.

4. OBSERVE: The agent reads the results. How many came back? Are the
   headlines actually about the topic?

5. REASON: flan-t5 evaluates quality and decides to accept, broaden,
   or refine the query. Rule-based heuristics provide a fallback if
   flan-t5 is unavailable.

6. ACT: If not accepted, the query is rewritten and the loop retries.
   Maximum 3 attempts before accepting whatever is available.

After the loop, VADER sentiment is applied and TextRank generates a
briefing. The full reasoning log is returned to the frontend and shown
in the sidebar so the user can see every decision the agent made.


In [ ]:
%%writefile /content/agent.py
from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass, field
from statistics import mean
from threading import Lock

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from fetch_news import fetch_news
from nlp_pipeline import assess_topic_specificity, generate_briefing
from sentiment import process_articles


FLAN_MODEL_NAME = "google/flan-t5-base"
TOOL_REGISTRY = {
    "search_news": "Fetches the latest Google News RSS results for a query.",
    "assess_quality": "Evaluates whether fetched headlines are numerous and relevant enough.",
    "refine_query": "Narrows the query when the fetched headlines drift away from the topic.",
    "broaden_query": "Broadens the query when too few articles are returned.",
    "analyze_sentiment": "Runs VADER sentiment classification on every headline.",
    "summarize": "Builds a short TextRank-style briefing for the accepted articles.",
}

STOPWORDS = {
    "the",
    "a",
    "an",
    "and",
    "or",
    "of",
    "for",
    "to",
    "in",
    "on",
    "at",
    "by",
    "from",
    "with",
    "latest",
    "news",
    "update",
    "updates",
    "today",
    "breaking",
}

VAGUE_EXPANSIONS = {
    "ai": "artificial intelligence latest news",
    "technology": "technology industry latest news",
    "economy": "global economy latest news",
    "markets": "stock market latest news",
    "sports": "sports latest news",
    "weather": "extreme weather latest news",
    "politics": "politics latest news",
    "business": "business latest news",
}

_flan_tokenizer = None
_flan_model = None
_flan_failed = False
_flan_lock = Lock()


def load_flan():
    global _flan_tokenizer, _flan_model, _flan_failed
    if _flan_failed:
        return None, None
    if _flan_tokenizer is not None and _flan_model is not None:
        return _flan_tokenizer, _flan_model
    with _flan_lock:
        if _flan_failed:
            return None, None
        if _flan_tokenizer is not None and _flan_model is not None:
            return _flan_tokenizer, _flan_model
        try:
            _flan_tokenizer = AutoTokenizer.from_pretrained(FLAN_MODEL_NAME)
            _flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL_NAME)
            return _flan_tokenizer, _flan_model
        except Exception:
            _flan_failed = True
            return None, None


def flan_reason(prompt: str, max_new_tokens: int = 32) -> str:
    tokenizer, model = load_flan()
    if tokenizer is None or model is None:
        return ""
    try:
        encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        generated = model.generate(**encoded, max_new_tokens=max_new_tokens)
        return tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    except Exception:
        return ""


def _tokenize(value: str) -> list[str]:
    return [
        token.lower()
        for token in re.findall(r"[A-Za-z0-9]+", value or "")
        if token.lower() not in STOPWORDS
    ]


def headline_relevance(headline: str, topic: str) -> float:
    topic_tokens = set(_tokenize(topic))
    headline_tokens = set(_tokenize(headline))
    if not topic_tokens:
        return 0.0
    overlap = len(topic_tokens & headline_tokens) / len(topic_tokens)
    if (topic or "").strip().lower() in (headline or "").strip().lower():
        overlap = max(overlap, 0.75)
    return round(overlap, 3)


def _extract_candidate_terms(articles: list[dict], topic: str) -> list[str]:
    topic_tokens = set(_tokenize(topic))
    counter = Counter()
    for article in articles or []:
        for token in _tokenize(article.get("headline", "")):
            if token not in topic_tokens and len(token) > 2:
                counter[token] += 1
    return [term for term, _count in counter.most_common(5)]


def _heuristic_expand_query(topic: str, specificity: str) -> str:
    lowered = (topic or "").strip().lower()
    if specificity in {"specific", "person"}:
        return topic
    return VAGUE_EXPANSIONS.get(lowered, f"{topic} latest news")


def agent_expand_query(topic: str, specificity: str) -> str:
    baseline = _heuristic_expand_query(topic, specificity)
    if specificity in {"specific", "person"}:
        return baseline

    prompt = f"""
    Rewrite this vague news topic into one concise search query for Google News.
    Topic: {topic}
    Keep it under 8 words and preserve the user's meaning.
    """
    candidate = flan_reason(prompt, max_new_tokens=20)
    if candidate:
        candidate = candidate.splitlines()[0].strip(" -:;")
    return candidate or baseline


def _heuristic_quality_decision(articles: list[dict], topic: str) -> dict:
    count = len(articles or [])
    relevance_scores = [headline_relevance(item.get("headline", ""), topic) for item in articles or []]
    avg_relevance = round(mean(relevance_scores), 3) if relevance_scores else 0.0

    if count == 0:
        decision = "broaden"
        reason = "No articles came back, so the agent needs a broader query."
    elif count < 3:
        decision = "broaden"
        reason = "Too few articles were returned, so the query should be broadened."
    elif avg_relevance < 0.16:
        decision = "refine"
        reason = "The headlines drift away from the topic, so the query should be refined."
    else:
        decision = "accept"
        reason = "The article count and relevance are strong enough to continue."

    return {
        "decision": decision,
        "reason": reason,
        "count": count,
        "avg_relevance": avg_relevance,
    }


def agent_assess_quality(articles: list[dict], topic: str) -> dict:
    heuristic = _heuristic_quality_decision(articles, topic)
    prompt = f"""
    You are assessing the quality of a news search.
    Topic: {topic}
    Articles found: {heuristic['count']}
    Average headline relevance: {heuristic['avg_relevance']}
    Choose one action: ACCEPT, BROADEN, or REFINE.
    Then give a short reason.
    """
    candidate = flan_reason(prompt, max_new_tokens=24).upper()
    if "ACCEPT" in candidate:
        heuristic["decision"] = "accept"
    elif "BROADEN" in candidate:
        heuristic["decision"] = "broaden"
    elif "REFINE" in candidate:
        heuristic["decision"] = "refine"

    if candidate:
        heuristic["reason"] = candidate.title()
    return heuristic


def _heuristic_refined_query(current_query: str, articles: list[dict], mode: str, original_topic: str) -> str:
    if mode == "broaden":
        if "latest news" not in current_query.lower():
            return f"{original_topic} latest news"
        parts = current_query.split()
        if len(parts) > 2:
            return " ".join(parts[:2] + ["news"])
        return current_query

    candidate_terms = _extract_candidate_terms(articles, original_topic)
    if candidate_terms:
        return f"{original_topic} {candidate_terms[0]} news"
    return f"{original_topic} latest updates"


def agent_refine_query(current_query: str, articles: list[dict], mode: str, original_topic: str) -> str:
    baseline = _heuristic_refined_query(current_query, articles, mode, original_topic)
    prompt = f"""
    Improve this Google News query.
    Original topic: {original_topic}
    Current query: {current_query}
    Mode: {mode}
    Sample headlines: {[item.get('headline', '') for item in articles[:5]]}
    Return only one short query under 10 words.
    """
    candidate = flan_reason(prompt, max_new_tokens=20)
    if candidate:
        candidate = candidate.splitlines()[0].strip(" -:;")
    return candidate or baseline


@dataclass
class VeritasAgent:
    max_attempts: int = 3
    reasoning_log: list[dict] = field(default_factory=list)

    def _log(self, stage: str, message: str, tool: str | None = None) -> None:
        entry = {"stage": stage, "message": message}
        if tool:
            entry["tool"] = tool
        self.reasoning_log.append(entry)

    def run(self, topic: str) -> dict:
        self.reasoning_log = []
        topic = (topic or "").strip()
        specificity = assess_topic_specificity(topic)
        self._log("observe", f"Classified topic specificity as {specificity}.")

        query = agent_expand_query(topic, specificity)
        if query == topic:
            self._log("reason", "Topic is already specific enough, so the first search uses it directly.")
        else:
            self._log(
                "reason",
                f"Topic is broad, so the agent expanded it to '{query}'.",
                tool="refine_query",
            )

        best_articles: list[dict] = []

        for attempt in range(1, self.max_attempts + 1):
            self._log(
                "act",
                f"Attempt {attempt}: searching Google News for '{query}'.",
                tool="search_news",
            )
            raw_articles = fetch_news(query, max_articles=15, display_topic=topic)
            self._log("observe", f"Retrieved {len(raw_articles)} candidate articles.")

            assessment = agent_assess_quality(raw_articles, topic)
            self._log(
                "reason",
                (
                    f"Quality decision: {assessment['decision']} "
                    f"(count={assessment['count']}, relevance={assessment['avg_relevance']}). "
                    f"{assessment['reason']}"
                ),
                tool="assess_quality",
            )

            best_articles = raw_articles
            if assessment["decision"] == "accept":
                break

            if attempt < self.max_attempts:
                next_query = agent_refine_query(
                    current_query=query,
                    articles=raw_articles,
                    mode=assessment["decision"],
                    original_topic=topic,
                )
                tool_name = "broaden_query" if assessment["decision"] == "broaden" else "refine_query"
                self._log(
                    "act",
                    f"Adjusting the query to '{next_query}' before the next retry.",
                    tool=tool_name,
                )
                query = next_query
            else:
                self._log(
                    "reason",
                    "Maximum retries reached, so the agent will continue with the best available set.",
                )

        processed = process_articles(best_articles, limit_per_topic=10)
        self._log(
            "act",
            f"Computed VADER sentiment for {len(processed)} accepted articles.",
            tool="analyze_sentiment",
        )
        self._log(
            "act",
            "Prepared the final briefing summary for this topic.",
            tool="summarize",
        )

        return {
            "topic": topic,
            "query": query,
            "articles": processed,
            "briefing": generate_briefing(processed),
            "reasoning": list(self.reasoning_log),
        }


def run_agent_for_topics(topics: list[str]) -> dict:
    clean_topics = [(topic or "").strip() for topic in topics or [] if (topic or "").strip()][:3]
    all_articles: list[dict] = []
    reasoning_log: dict[str, list[dict]] = {}
    topic_briefings: dict[str, str] = {}

    for topic in clean_topics:
        agent = VeritasAgent(max_attempts=3)
        result = agent.run(topic)
        all_articles.extend(result["articles"])
        reasoning_log[topic] = result["reasoning"]
        topic_briefings[topic] = result["briefing"]

    all_articles.sort(key=lambda item: item.get("timestamp", 0), reverse=True)
    return {
        "articles": all_articles,
        "briefing": generate_briefing(all_articles),
        "topics": clean_topics,
        "reasoning_log": reasoning_log,
        "topic_briefings": topic_briefings,
        "tool_registry": TOOL_REGISTRY,
    }


## Step 8 — VeritasBot Chatbot Module

VeritasBot is the conversational layer that sits on top of everything
the agent has already built. Once news is loaded the user can have a
natural conversation about it or ask for new searches.

Six enhancements are built on top of a base news search chatbot:

Enhancement 1: Natural language query understanding via spaCy NER. The
user can type anything and the bot extracts the topic. "What is happening
with Apple lately?" works the same as "Find news on Apple."

Enhancement 2: Multi-turn memory. The bot tracks the last topic so
follow-up messages like "what about their stock price?" resolve correctly
against the previous context.

Enhancement 3: Sentiment-aware response tone. The bot checks the
sentiment breakdown of what it finds and adjusts its opening sentence
accordingly. Mostly negative coverage gets a different opener than
mostly positive.

Enhancement 4: Suggested follow-up chips. After every response the bot
generates 2 to 3 clickable topic buttons from entities in the fetched
headlines. The user clicks a chip and it auto-sends as the next message.

Enhancement 5: Briefing mode. When the user asks for a briefing or
summary, TextRank runs and returns a 2 to 3 sentence digest.

Enhancement 6: Sentiment percentage commentary. Every response ends
with an exact breakdown like 60% Positive, 30% Negative, 10% Neutral.

The chatbot returns structured data with article objects so the frontend
can render proper clickable cards with colored borders, not plain URLs.


In [ ]:
%%writefile /content/chatbot.py
from __future__ import annotations

import re

from fetch_news import fetch_news
from nlp_pipeline import (
    extract_entities_from_query,
    extract_followup_suggestions,
    generate_briefing,
)
from sentiment import process_articles, sentiment_breakdown


FOLLOWUP_PHRASES = {
    "stock": "stock price",
    "price": "stock price",
    "shares": "stock price",
    "earnings": "earnings",
    "guidance": "guidance",
    "lawsuit": "lawsuit",
    "acquisition": "acquisition",
    "flood": "flood updates",
    "weather": "weather impact",
}


def get_sentiment_tone(articles: list[dict]) -> str:
    breakdown = sentiment_breakdown(articles)
    positives = breakdown["percentages"].get("Positive", 0.0)
    negatives = breakdown["percentages"].get("Negative", 0.0)
    if negatives > positives + 15:
        return "The overall tone in the latest coverage is leaning negative right now."
    if positives > negatives + 15:
        return "The overall tone in the latest coverage is leaning positive right now."
    return "The latest coverage is fairly mixed right now."


def is_briefing_request(message: str) -> bool:
    lowered = (message or "").lower()
    triggers = {"brief", "briefing", "summary", "summarize", "recap", "what is happening"}
    return any(trigger in lowered for trigger in triggers)


def _format_breakdown_line(articles: list[dict]) -> str:
    breakdown = sentiment_breakdown(articles)
    percentages = breakdown["percentages"]
    return (
        f"{percentages.get('Positive', 0.0)}% Positive, "
        f"{percentages.get('Negative', 0.0)}% Negative, "
        f"{percentages.get('Neutral', 0.0)}% Neutral."
    )


def format_article_list(articles: list[dict], limit: int = 3) -> list[dict]:
    rendered = []
    for article in articles[:limit]:
        rendered.append(
            {
                "headline": article.get("headline", ""),
                "translated_headline": article.get("translated_headline", article.get("headline", "")),
                "link": article.get("link", ""),
                "source": article.get("source", ""),
                "published_display": article.get("published_display", ""),
                "sentiment": article.get("sentiment", "Neutral"),
                "topic": article.get("topic", ""),
            }
        )
    return rendered


def _clean_query_text(message: str) -> str:
    cleaned = re.sub(
        r"(?i)\b(find|show|give|tell|latest|news|on|about|what|is|going|with|please|me|the)\b",
        " ",
        message or "",
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ?.!,:;")
    return cleaned


def _resolve_from_history(history: list[dict]) -> str:
    for item in reversed(history or []):
        content = item.get("content", "")
        entities = extract_entities_from_query(content)
        if entities:
            return entities[0]
    return ""


def resolve_topic(message: str, history: list[dict] | None = None, last_topic: str | None = None) -> str:
    entities = extract_entities_from_query(message)
    if entities:
        return entities[0]

    lowered = (message or "").lower()
    suffixes = [value for key, value in FOLLOWUP_PHRASES.items() if key in lowered]
    if last_topic and (suffixes or re.search(r"\b(it|their|them|that|those|he|she|its)\b", lowered)):
        suffix = suffixes[0] if suffixes else ""
        return f"{last_topic} {suffix}".strip()

    cleaned = _clean_query_text(message)
    if cleaned:
        return cleaned

    if last_topic:
        return last_topic

    return _resolve_from_history(history or [])


def generate_chatbot_response(
    message: str,
    history: list[dict] | None = None,
    last_topic: str | None = None,
    articles: list[dict] | None = None,
) -> dict:
    history = history or []
    current_articles = articles or []
    active_topic = resolve_topic(message, history=history, last_topic=last_topic)

    if is_briefing_request(message) and current_articles:
        tone = get_sentiment_tone(current_articles)
        briefing = generate_briefing(current_articles, sentence_count=3)
        suggestions = extract_followup_suggestions(current_articles, active_topic or last_topic or "")
        return {
            "response": f"{tone} {briefing} {_format_breakdown_line(current_articles)}",
            "last_topic": active_topic or last_topic,
            "suggestions": suggestions,
            "articles": format_article_list(current_articles, limit=3),
            "mode": "briefing",
            "sentiment_commentary": _format_breakdown_line(current_articles),
        }

    if active_topic:
        fresh_articles = process_articles(fetch_news(active_topic, max_articles=10), limit_per_topic=10)
    else:
        fresh_articles = process_articles(current_articles, limit_per_topic=10) if current_articles else []

    if not fresh_articles and current_articles:
        fresh_articles = process_articles(current_articles, limit_per_topic=10)

    if not fresh_articles:
        return {
            "response": "I could not find fresh articles for that topic yet, but you can try a more specific news query next.",
            "last_topic": active_topic or last_topic,
            "suggestions": [],
            "articles": [],
            "mode": "fallback",
            "sentiment_commentary": "0.0% Positive, 0.0% Negative, 0.0% Neutral.",
        }

    tone = get_sentiment_tone(fresh_articles)
    article_cards = format_article_list(fresh_articles, limit=3)
    suggestions = extract_followup_suggestions(fresh_articles, active_topic or last_topic or "")
    sentiment_line = _format_breakdown_line(fresh_articles)
    response = (
        f"{tone} I found recent coverage for {active_topic or last_topic or 'this topic'}. "
        f"Here are a few links to open next. {sentiment_line}"
    )

    return {
        "response": response,
        "last_topic": active_topic or last_topic,
        "suggestions": suggestions,
        "articles": article_cards,
        "mode": "articles",
        "sentiment_commentary": sentiment_line,
    }


## Step 9 — Frontend Template

This cell writes the complete HTML, CSS, and JavaScript frontend to a
file called templates.py. The entire frontend is embedded as a Python
raw string so everything stays inside one Colab file.

The frontend is a custom news website design with:
- Clean white background and bold typography
- Top navbar with centered logo and category navigation
- Search section with three topic inputs and a language dropdown
- Briefing card that appears after the first fetch
- Featured hero article on the left with large bold headline
- Secondary articles listed on the right with thumbnail placeholders
- Insights section with word cloud and three Plotly charts
- VeritasBot chat panel with message bubbles and suggestion chips
- Agent reasoning log showing every step the agent took

All state is managed in JavaScript variables. The frontend calls the
FastAPI endpoints using the browser's built-in fetch API with no
external JavaScript framework needed.


In [ ]:
%%writefile /content/templates.py
HTML_PAGE = r"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <meta http-equiv="Cache-Control" content="no-cache, no-store, must-revalidate" />
  <meta http-equiv="Pragma" content="no-cache" />
  <meta http-equiv="Expires" content="0" />
  <title>VeritasAI</title>
  <link rel="icon" href="data:image/svg+xml,<svg xmlns=%22http://www.w3.org/2000/svg%22 viewBox=%220 0 100 100%22><circle cx=%2250%22 cy=%2250%22 r=%2248%22 fill=%22%232D8A70%22/><text y=%22.9em%22 x=%2250%%22 font-size=%2270%22 text-anchor=%22middle%22 fill=%22white%22 font-family=%22serif%22 font-weight=%22bold%22>V</text></svg>"> 
  <link rel="preconnect" href="https://fonts.googleapis.com" />
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin />
  <link href="https://fonts.googleapis.com/css2?family=Montserrat:ital,wght@1,500;1,600&display=swap" rel="stylesheet">
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@800&family=Playfair+Display:ital,wght@1,900&display=swap" rel="stylesheet">
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&family=Newsreader:opsz,wght@6..72,600;6..72,700;6..72,800&display=swap" rel="stylesheet" />
  <script>
    (function () {
      function keepWarm() {
        fetch("/", { cache: "no-store" }).catch(function () {});
      }
      keepWarm();
      window.__veritasKeepWarm = window.setInterval(keepWarm, 45000);
    })();
  </script>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    :root {
      color-scheme: light;
      --bg-color: #FDFCF8;
      --text-color: #1A1A1A;
      --card-shadow: rgba(0, 0, 0, 0.04);
      --card-bg: #FFFFFF;
      --bg: var(--bg-color);
      --surface: var(--card-bg);
      --surface-soft: #F4F0E7;
      --surface-muted: #ECE7DB;
      --text: var(--text-color);
      --muted: #566257;
      --muted-soft: #768274;
      --border: rgba(23, 33, 25, 0.10);
      --accent: #215E46;
      --accent-bright: #2D8A70;
      --accent-dark: #164633;
      --accent-soft: #E1EEE7;
      --positive-bg: #ECF8F1;
      --positive-text: #20643F;
      --positive-border: #BCDCC8;
      --negative-bg: #FBEFED;
      --negative-text: #A63B32;
      --negative-border: #E8C0BA;
      --neutral-bg: #F1F0EC;
      --neutral-text: #6A726C;
      --neutral-border: #DFE2DC;
      --shadow-sm: 0 16px 40px var(--card-shadow);
      --shadow-lg: 0 30px 72px rgba(0, 0, 0, 0.08);
      --page-radial: rgba(45, 138, 112, 0.06);
      --page-bg-start: #FDFCF8;
      --page-bg-end: #FDFCF8;
      --hero-radial-strong: rgba(45, 138, 112, 0.10);
      --hero-radial-soft: rgba(45, 138, 112, 0.06);
      --hero-bg-start: #FDFCF8;
      --hero-bg-end: #FDFCF8;
      --hero-terminal-bg: linear-gradient(180deg, rgba(255, 255, 255, 0.9), rgba(252, 249, 243, 0.84));
      --hero-terminal-accent: rgba(45, 138, 112, 0.08);
      --blank-card-bg: #FFFFFF;
      --blank-card-line: #EFF2EE;
      --blank-chip-bg: #F3F5F3;
      --blank-card-box-bg: #FAFBFA;
      --ghost-bg: rgba(255, 255, 255, 0.76);
      --ghost-hover-bg: #FFFFFF;
      --ticker-badge-bg: #1A1A1A;
      --ticker-badge-text: #FFFFFF;
      --warm-status: #B45309;
      --chart-grid: rgba(0, 0, 0, 0.10);
      --chart-zero: rgba(0, 0, 0, 0.16);
      --chart-band: rgba(45, 138, 112, 0.10);
      --chart-marker: rgba(33, 94, 70, 0.72);
      --chart-marker-line: rgba(255, 255, 255, 0.92);
      --chart-legend: #1A1A1A;
      --max-width: 1440px;
    }

    * {
      box-sizing: border-box;
    }

    html {
      scroll-behavior: smooth;
    }

    body[data-theme="dark"] {
      color-scheme: dark;
      --bg-color: #121212;
      --text-color: #F5F5F5;
      --card-shadow: rgba(0, 0, 0, 0.4);
      --card-bg: #1E1E1E;
      --bg: var(--bg-color);
      --surface: var(--card-bg);
      --surface-soft: #1A1A1A;
      --surface-muted: #222222;
      --text: var(--text-color);
      --muted: #B5C3B8;
      --muted-soft: #8F9E92;
      --border: rgba(218, 232, 221, 0.10);
      --accent: #66B397;
      --accent-bright: #2D8A70;
      --accent-dark: #4B9A7E;
      --accent-soft: rgba(45, 138, 112, 0.16);
      --positive-bg: rgba(45, 138, 112, 0.16);
      --positive-text: #C5F1D5;
      --positive-border: rgba(103, 183, 133, 0.28);
      --negative-bg: rgba(169, 68, 66, 0.16);
      --negative-text: #F7C4C3;
      --negative-border: rgba(169, 68, 66, 0.26);
      --neutral-bg: rgba(112, 128, 144, 0.14);
      --neutral-text: #D4DBE2;
      --neutral-border: rgba(112, 128, 144, 0.22);
      --shadow-sm: 0 16px 40px var(--card-shadow);
      --shadow-lg: 0 30px 72px rgba(0, 0, 0, 0.55);
      --page-radial: rgba(45, 138, 112, 0.12);
      --page-bg-start: #121212;
      --page-bg-end: #171717;
      --hero-radial-strong: rgba(45, 138, 112, 0.16);
      --hero-radial-soft: rgba(45, 138, 112, 0.08);
      --hero-bg-start: #121212;
      --hero-bg-end: #171717;
      --hero-terminal-bg: linear-gradient(180deg, rgba(24, 24, 24, 0.94), rgba(18, 18, 18, 0.92));
      --hero-terminal-accent: rgba(45, 138, 112, 0.12);
      --blank-card-bg: #1A1A1A;
      --blank-card-line: rgba(219, 231, 223, 0.10);
      --blank-chip-bg: rgba(219, 231, 223, 0.08);
      --blank-card-box-bg: #222222;
      --ghost-bg: rgba(24, 24, 24, 0.76);
      --ghost-hover-bg: rgba(31, 31, 31, 0.94);
      --ticker-badge-bg: #F1F5F1;
      --ticker-badge-text: #132018;
      --warm-status: #EAB308;
      --chart-grid: rgba(219, 231, 223, 0.10);
      --chart-zero: rgba(219, 231, 223, 0.18);
      --chart-band: rgba(45, 138, 112, 0.18);
      --chart-marker: rgba(102, 179, 151, 0.76);
      --chart-marker-line: rgba(15, 21, 17, 0.92);
      --chart-legend: #D8E0DA;
    }

    body {
      margin: 0;
      background:
        radial-gradient(circle at top left, var(--page-radial), transparent 24%),
        linear-gradient(180deg, var(--page-bg-start) 0%, var(--page-bg-end) 100%);
      color: var(--text);
      font-family: "Inter", sans-serif;
      -webkit-font-smoothing: antialiased;
      text-rendering: optimizeLegibility;
      transition: background 0.28s ease, color 0.28s ease;
    }

    a {
      color: inherit;
      text-decoration: none;
    }

    button,
    input,
    select {
      font: inherit;
    }

    button {
      cursor: pointer;
    }

    .shell {
      max-width: var(--max-width);
      margin: 0 auto;
      padding: 0 28px 40px;
    }

    .surface {
      background-color: var(--card-bg);
      border: 1px solid var(--border);
      border-radius: 1.5rem;
      box-shadow: var(--shadow-sm);
      backdrop-filter: blur(18px);
    }

    .interactive {
      transition: transform 0.22s ease, box-shadow 0.22s ease, border-color 0.22s ease, background-color 0.22s ease;
    }

    .interactive:hover {
      transform: translateY(-4px);
      box-shadow: var(--shadow-lg);
    }

    .hero {
      position: relative;
      overflow: hidden;
      padding: 60px 28px;
      background:
        radial-gradient(circle at top right, var(--hero-radial-strong), transparent 30%),
        radial-gradient(circle at left center, var(--hero-radial-soft), transparent 28%),
        linear-gradient(180deg, var(--hero-bg-start) 0%, var(--hero-bg-end) 100%);
      border-bottom: 1px solid rgba(23, 33, 25, 0.08);
    }

    .hero-inner {
      position: relative;
      z-index: 2;
      max-width: 1040px;
      margin: 0 auto;
      text-align: center;
    }

    .theme-toggle {
      min-height: 48px;
      padding: 0 14px;
      border-radius: 999px;
      border: 1px solid var(--border);
      background: var(--ghost-bg);
      color: var(--muted);
      font-size: 0.78rem;
      font-weight: 700;
      letter-spacing: 0.02em;
      display: inline-flex;
      align-items: center;
      justify-content: center;
      backdrop-filter: blur(10px);
      transition: transform 0.2s ease, border-color 0.2s ease, background-color 0.2s ease, color 0.2s ease;
    }

    .theme-toggle:hover {
      transform: translateY(-1px);
      color: var(--accent);
      border-color: rgba(33, 94, 70, 0.24);
      background: var(--ghost-hover-bg);
    }

    .logo-wrapper {
      display: flex;
      align-items: baseline;
      justify-content: center;
      gap: 0.08rem;
      font-size: clamp(3rem, 8vw, 5.2rem);
      line-height: 1;
    }

    .veritas-serif {
      font-family: "Playfair Display", serif;
      font-style: italic;
      font-weight: 900;
      color: var(--text);
      position: relative;
      letter-spacing: -0.02em;
      line-height: 0.92;
    }

    .veritas-i {
      position: relative;
      display: inline-block;
    }

    .veritas-i::after {
      content: "";
      position: absolute;
      left: 50%;
      top: -4px;
      transform: translateX(-50%);
      width: 8px;
      height: 8px;
      background: #2D8A70;
      border-radius: 50%;
      box-shadow:
        0 0 0 1px rgba(255, 255, 255, 0.45),
        0 0 10px rgba(45, 138, 112, 0.4),
        0 0 20px rgba(45, 138, 112, 0.16);
    }

    .ai-sans {
      font-family: "Inter", sans-serif;
      font-weight: 800;
      color: var(--text);
      font-size: 0.8em;
      letter-spacing: 0.05em;
      margin-left: 0.08em;
      text-transform: uppercase;
    }

    .hero-tagline {
      margin: 12px 0 0;
      color: var(--accent);
      font-size: 0.78rem;
      font-weight: 800;
      letter-spacing: 0.28em;
      text-transform: uppercase;
      animation: trackingInExpand 0.95s cubic-bezier(0.2, 0.8, 0.2, 1) both 0.12s;
    }

    .hero-copy {
      max-width: 780px;
      margin: 0.9rem auto 0;
      color: var(--muted);
      font-size: 1rem;
      line-height: 1.86;
    }

    .hero-actions {
      display: flex;
      align-items: center;
      justify-content: center;
      flex-wrap: wrap;
      gap: 12px;
      margin-top: 10px;
    }

    .hero-actions .control {
      width: auto;
      min-width: 220px;
      flex: 0 1 220px;
      box-shadow: none;
    }

    .hero-actions > * {
      align-self: center;
    }

    .hero-visuals {
      position: relative;
      width: min(960px, 100%);
      min-height: 440px;
      margin: 28px auto 0;
      padding: 0.5rem 0 1rem;
      overflow: visible;
    }

    .blank-stack {
      position: relative;
      width: min(760px, 100%);
      height: 390px;
      margin: 0 auto;
      perspective: 1600px;
    }

    .synthesis-progress {
      position: absolute;
      inset: 0;
      display: flex;
      align-items: center;
      justify-content: center;
      padding: 1rem 0;
    }

    .synthesis-progress[hidden] {
      display: none;
    }

    .synthesis-progress-shell {
      width: min(560px, calc(100% - 32px));
      min-height: 240px;
      display: grid;
      align-content: center;
      justify-items: center;
      gap: 16px;
      padding: 2.2rem 2rem;
      border-radius: 1.75rem;
      border: 1px solid var(--border);
      background:
        var(--hero-terminal-bg),
        radial-gradient(circle at top right, var(--hero-terminal-accent), transparent 42%);
      box-shadow: 0 28px 62px var(--card-shadow);
      backdrop-filter: blur(14px);
    }

    .synthesis-progress-title {
      margin: 0;
      color: var(--text);
      font-size: 0.84rem;
      font-weight: 800;
      letter-spacing: 0.18em;
      text-transform: uppercase;
    }

    .synthesis-progress-track {
      position: relative;
      width: min(340px, 100%);
      height: 3px;
      border-radius: 999px;
      background: rgba(45, 138, 112, 0.16);
      overflow: hidden;
    }

    .synthesis-progress-track::after {
      content: "";
      position: absolute;
      top: 0;
      bottom: 0;
      width: 38%;
      border-radius: inherit;
      background: linear-gradient(90deg, rgba(45, 138, 112, 0), rgba(45, 138, 112, 0.95), rgba(45, 138, 112, 0));
      animation: synthesisPulse 1.9s ease-in-out infinite;
    }

    .synthesis-progress-message {
      max-width: 430px;
      margin: 0;
      min-height: 3.2rem;
      color: color-mix(in srgb, var(--accent) 82%, var(--text) 18%);
      font-family: "Montserrat", sans-serif;
      font-style: italic;
      font-size: 0.9rem;
      font-weight: 600;
      line-height: 1.7;
      letter-spacing: 0.01em;
      text-align: center;
      opacity: 0.88;
    }

    .synthesis-progress-cancel {
      min-height: 40px;
      padding: 0 14px;
      border-radius: 999px;
      border: 1px solid var(--border);
      background: transparent;
      color: var(--muted);
      font-size: 0.8rem;
      font-weight: 700;
      letter-spacing: 0.01em;
    }

    .synthesis-progress-cancel:hover {
      color: var(--accent);
      border-color: rgba(33, 94, 70, 0.26);
      background: var(--ghost-bg);
    }

    .blank-card {
      --card-transform: translateY(0);
      --card-transform-hidden: translateY(36px) scale(0.985);
      --card-transform-loaded: var(--card-transform);
      --card-transform-hover: var(--card-transform-loaded);
      --card-transform-focus: var(--card-transform-hover);
      position: absolute;
      width: min(430px, calc(100% - 40px));
      min-height: 245px;
      padding: 1.45rem;
      border-radius: 1.5rem;
      border: 1px solid rgba(23, 33, 25, 0.06);
      background: var(--blank-card-bg);
      box-shadow: 0 40px 80px var(--card-shadow);
      opacity: 0;
      transform: var(--card-transform-hidden);
      transform-origin: center center;
      transition: all 0.5s cubic-bezier(0.16, 1, 0.3, 1);
      will-change: transform, opacity, box-shadow;
      overflow: hidden;
    }

    body.is-ready .blank-card {
      opacity: 1;
      transform: var(--card-transform);
    }

    .blank-card::before {
      content: "";
      position: absolute;
      inset: 0;
      background:
        linear-gradient(180deg, rgba(246, 247, 245, 0.28), rgba(255, 255, 255, 0)),
        radial-gradient(circle at top right, rgba(45, 138, 112, 0.06), transparent 35%);
      opacity: 1;
      transition: opacity 0.35s ease, background 0.35s ease;
      pointer-events: none;
    }

    .blank-card::after {
      content: "";
      position: absolute;
      left: 0;
      top: 0;
      width: 100%;
      height: 3px;
      background: #2D8A70;
      opacity: 0;
      transition: opacity 0.5s cubic-bezier(0.16, 1, 0.3, 1);
      pointer-events: none;
    }

    .blank-card > * {
      position: relative;
      z-index: 1;
    }

    .blank-card--one {
      --card-transform: translateX(calc(-50% + 42px)) rotate(4deg) translateY(30px);
      --card-transform-hidden: translateX(calc(-50% + 42px)) rotate(4deg) translateY(58px) scale(0.985);
      --card-transform-loaded: translateX(calc(-50% + 62px)) rotate(4deg) translateY(20px);
      --card-transform-hover: translateX(calc(-50% + 320px)) rotate(2deg) translateY(4px);
      --card-transform-focus: translateX(calc(-50% + 320px)) rotate(2deg) translateY(4px) scale(1.05);
      left: 50%;
      bottom: 22px;
      z-index: 1;
      transition-delay: 0s;
    }

    .blank-card--two {
      --card-transform: translateX(calc(-50% - 42px)) rotate(-4deg) translateY(30px);
      --card-transform-hidden: translateX(calc(-50% - 42px)) rotate(-4deg) translateY(58px) scale(0.985);
      --card-transform-loaded: translateX(calc(-50% - 62px)) rotate(-4deg) translateY(20px);
      --card-transform-hover: translateX(calc(-50% - 320px)) rotate(-2deg) translateY(4px);
      --card-transform-focus: translateX(calc(-50% - 320px)) rotate(-2deg) translateY(4px) scale(1.05);
      left: 50%;
      bottom: 22px;
      z-index: 2;
      transition-delay: 0.15s;
    }

    .blank-card--three {
      --card-transform: translateX(-50%) translateY(0);
      --card-transform-hidden: translateX(-50%) translateY(44px) scale(0.985);
      --card-transform-loaded: translateX(-50%) translateY(-2px) scale(1);
      --card-transform-hover: translateX(-50%) translateY(-10px) scale(1);
      --card-transform-focus: translateX(-50%) translateY(-10px) scale(1.05);
      left: 50%;
      top: 0;
      width: min(500px, calc(100% - 28px));
      min-height: 300px;
      z-index: 3;
      transition-delay: 0.3s;
    }

    .blank-stack.is-loaded .blank-card {
      transform: var(--card-transform-loaded);
    }

    .blank-stack.is-loaded .blank-card--one {
      transition-delay: 0s;
    }

    .blank-stack.is-loaded .blank-card--two {
      transition-delay: 0.08s;
    }

    .blank-stack.is-loaded .blank-card--three {
      transition-delay: 0.16s;
    }

    .blank-stack.is-loaded .blank-card::before {
      opacity: 0.92;
    }

    @media (hover: hover) and (pointer: fine) {
      .blank-stack.is-loaded:hover .blank-card.is-live {
        transform: var(--card-transform-hover);
        opacity: 1;
        background: #FFFFFF;
        box-shadow: 0 34px 86px rgba(23, 33, 25, 0.18);
        transition-delay: 0s;
      }

      .blank-stack.is-loaded:hover .blank-card.is-live::before {
        opacity: 0.96;
      }

      .blank-stack.is-loaded:hover .blank-card--one.is-live,
      .blank-stack.is-loaded:hover .blank-card--two.is-live {
        box-shadow: 0 30px 76px rgba(23, 33, 25, 0.16);
      }

      .blank-stack.is-loaded:hover .blank-card--one.is-live::after,
      .blank-stack.is-loaded:hover .blank-card--two.is-live::after {
        opacity: 1;
      }

      .blank-stack.is-loaded:hover .blank-card--three.is-live {
        box-shadow: 0 40px 94px rgba(23, 33, 25, 0.20);
      }

      .blank-stack.is-loaded:hover:has(.blank-card.is-live:hover) .blank-card.is-live {
        opacity: 0.6;
      }

      .blank-stack.is-loaded:hover:has(.blank-card.is-live:hover) .blank-card.is-live:hover {
        opacity: 1;
        z-index: 999;
        transform: var(--card-transform-focus);
        box-shadow: 0 44px 108px rgba(23, 33, 25, 0.26);
      }

      body[data-theme="dark"] .blank-stack.is-loaded:hover .blank-card.is-live {
        background: #1E1E1E;
        box-shadow: 0 34px 88px rgba(0, 0, 0, 0.54);
      }

      body[data-theme="dark"] .blank-stack.is-loaded:hover .blank-card--one.is-live,
      body[data-theme="dark"] .blank-stack.is-loaded:hover .blank-card--two.is-live {
        box-shadow: 0 30px 80px rgba(0, 0, 0, 0.50);
      }

      body[data-theme="dark"] .blank-stack.is-loaded:hover .blank-card--three.is-live {
        box-shadow: 0 40px 96px rgba(0, 0, 0, 0.56);
      }

      body[data-theme="dark"] .blank-stack.is-loaded:hover:has(.blank-card.is-live:hover) .blank-card.is-live:hover {
        background: #1E1E1E;
        box-shadow:
          0 44px 110px rgba(0, 0, 0, 0.62),
          0 0 20px rgba(45, 138, 112, 0.14);
      }
    }

    @media (hover: none), (pointer: coarse) {
      .blank-card--one {
        --card-transform-loaded: translateX(calc(-50% + 88px)) rotate(4deg) translateY(18px);
      }

      .blank-card--two {
        --card-transform-loaded: translateX(calc(-50% - 88px)) rotate(-4deg) translateY(18px);
      }

      .blank-card--three {
        --card-transform-loaded: translateX(-50%) translateY(-6px) scale(1);
      }
    }

    .blank-card.is-live {
      display: flex;
      flex-direction: column;
      justify-content: space-between;
      gap: 1rem;
      background: #FFFFFF;
      border-color: rgba(23, 33, 25, 0.08);
      color: var(--text);
      box-shadow: 0 28px 70px rgba(23, 33, 25, 0.11);
    }

    .blank-card.is-live::before {
      background:
        radial-gradient(circle at top right, rgba(45, 138, 112, 0.10), transparent 34%),
        linear-gradient(180deg, rgba(255, 255, 255, 0.32), rgba(255, 255, 255, 0));
    }

    .hero-card-shell {
      display: flex;
      flex-direction: column;
      gap: 1rem;
      min-height: 100%;
    }

    .hero-card-meta {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 0.75rem;
      flex-wrap: wrap;
    }

    .hero-card-kicker,
    .hero-card-stamp {
      display: inline-flex;
      align-items: center;
      min-height: 28px;
      padding: 0 10px;
      border-radius: 999px;
      border: 1px solid rgba(23, 33, 25, 0.08);
      background: rgba(255, 255, 255, 0.72);
      color: var(--accent);
      font-size: 0.68rem;
      font-weight: 800;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .hero-card-stamp {
      color: var(--muted);
    }

    .hero-card-title {
      margin: 0;
      font-family: "Newsreader", Georgia, serif;
      font-size: clamp(1.9rem, 3vw, 2.8rem);
      font-weight: 700;
      line-height: 1.02;
      letter-spacing: -0.04em;
      text-wrap: balance;
    }

    .hero-card-summary {
      margin: 0;
      color: var(--muted);
      font-size: 0.95rem;
      line-height: 1.72;
    }

    .hero-card-footer {
      display: flex;
      align-items: flex-end;
      justify-content: space-between;
      gap: 1rem;
      margin-top: auto;
    }

    .hero-card-caption {
      margin: 0;
      color: var(--muted);
      font-size: 0.76rem;
      line-height: 1.6;
      max-width: 18rem;
    }

    .hero-card-arrow {
      width: 40px;
      height: 40px;
      flex-shrink: 0;
      display: inline-flex;
      align-items: center;
      justify-content: center;
      border-radius: 999px;
      border: 1px solid rgba(23, 33, 25, 0.10);
      background: rgba(255, 255, 255, 0.84);
      color: var(--accent);
      box-shadow: 0 10px 24px rgba(23, 33, 25, 0.08);
      animation: heroArrowPulse 2s ease-in-out infinite;
    }

    .hero-card-arrow:hover {
      transform: translateY(2px);
    }

    .hero-card-arrow svg {
      width: 16px;
      height: 16px;
      stroke: currentColor;
      fill: none;
      stroke-width: 2;
      stroke-linecap: round;
      stroke-linejoin: round;
    }

    .hero-card-list {
      display: flex;
      flex-direction: column;
      gap: 0.72rem;
    }

    .hero-card-list-label {
      margin: 0;
      color: var(--muted);
      font-size: 0.72rem;
      font-weight: 800;
      letter-spacing: 0.14em;
      text-transform: uppercase;
    }

    .hero-card-badge-row {
      display: flex;
      flex-wrap: wrap;
      gap: 0.55rem;
    }

    .hero-card-badge {
      display: inline-flex;
      align-items: center;
      min-height: 34px;
      padding: 0 12px;
      border-radius: 999px;
      border: 1px solid rgba(23, 33, 25, 0.10);
      background: rgba(45, 138, 112, 0.08);
      color: var(--text);
      font-size: 0.74rem;
      font-weight: 700;
      letter-spacing: 0.02em;
    }

    .hero-card-note {
      margin: 0;
      color: var(--muted);
      font-size: 0.84rem;
      line-height: 1.68;
    }

    .hero-card-stat-grid {
      display: grid;
      grid-template-columns: repeat(2, minmax(0, 1fr));
      gap: 0.85rem;
    }

    .hero-card-stat {
      padding: 0.9rem;
      border-radius: 1rem;
      border: 1px solid rgba(23, 33, 25, 0.08);
      background: rgba(255, 255, 255, 0.76);
    }

    .hero-card-stat-label {
      margin: 0;
      color: var(--muted);
      font-size: 0.68rem;
      font-weight: 800;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .hero-card-stat-value {
      margin: 0.4rem 0 0;
      color: var(--text);
      font-family: "Newsreader", Georgia, serif;
      font-size: 1.35rem;
      font-weight: 700;
      letter-spacing: -0.03em;
    }

    .hero-card-stat-copy {
      margin: 0.35rem 0 0;
      color: var(--muted);
      font-size: 0.76rem;
      line-height: 1.55;
    }

    body[data-theme="dark"] .blank-card.is-live {
      background: #1E1E1E;
      border-color: rgba(102, 179, 151, 0.42);
      box-shadow:
        0 0 0 1px rgba(102, 179, 151, 0.14),
        0 24px 64px rgba(0, 0, 0, 0.44),
        0 0 26px rgba(102, 179, 151, 0.12);
      color: #FFFFFF;
    }

    body[data-theme="dark"] .blank-card.is-live::before {
      background:
        radial-gradient(circle at top right, rgba(102, 179, 151, 0.16), transparent 34%),
        linear-gradient(180deg, rgba(255, 255, 255, 0.04), rgba(255, 255, 255, 0));
    }

    body[data-theme="dark"] .hero-card-kicker,
    body[data-theme="dark"] .hero-card-stamp,
    body[data-theme="dark"] .hero-card-badge,
    body[data-theme="dark"] .hero-card-stat {
      background: rgba(255, 255, 255, 0.04);
      border-color: rgba(255, 255, 255, 0.10);
    }

    body[data-theme="dark"] .hero-card-kicker,
    body[data-theme="dark"] .hero-card-arrow {
      color: #8FD3B8;
    }

    body[data-theme="dark"] .hero-card-stamp,
    body[data-theme="dark"] .hero-card-summary,
    body[data-theme="dark"] .hero-card-caption,
    body[data-theme="dark"] .hero-card-list-label,
    body[data-theme="dark"] .hero-card-note,
    body[data-theme="dark"] .hero-card-stat-label,
    body[data-theme="dark"] .hero-card-stat-copy {
      color: rgba(245, 245, 245, 0.78);
    }

    body[data-theme="dark"] .hero-card-title,
    body[data-theme="dark"] .hero-card-stat-value,
    body[data-theme="dark"] .hero-card-badge {
      color: #FFFFFF;
    }

    body[data-theme="dark"] .hero-card-arrow {
      background: rgba(255, 255, 255, 0.05);
      border-color: rgba(102, 179, 151, 0.36);
      box-shadow: 0 10px 26px rgba(0, 0, 0, 0.32), 0 0 18px rgba(102, 179, 151, 0.14);
    }

    .blank-card-top {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 10px;
    }

    .blank-pill {
      height: 11px;
      border-radius: 999px;
      background: var(--blank-line);
    }

    .blank-pill.short {
      width: 74px;
    }

    .blank-pill.mid {
      width: 96px;
    }

    .blank-pill.long {
      width: 132px;
    }

    .blank-chip-row {
      display: flex;
      gap: 8px;
      align-items: center;
    }

    .blank-chip {
      width: 32px;
      height: 10px;
      border-radius: 999px;
      background: var(--blank-chip-bg);
    }

    .blank-line {
      height: 12px;
      margin-top: 12px;
      border-radius: 999px;
      background: var(--blank-line);
    }

    .blank-line.xl {
      width: 88%;
      height: 20px;
      margin-top: 20px;
    }

    .blank-line.lg {
      width: 78%;
      height: 16px;
    }

    .blank-line.md {
      width: 70%;
    }

    .blank-line.sm {
      width: 48%;
    }

    .blank-card-frame {
      display: grid;
      grid-template-columns: repeat(2, minmax(0, 1fr));
      gap: 12px;
      margin-top: 22px;
    }

    .blank-card-box {
      min-height: 92px;
      border-radius: 1.15rem;
      border: 1px solid rgba(23, 33, 25, 0.06);
      background: var(--blank-card-box-bg);
    }

    .blank-card-footer {
      display: flex;
      align-items: center;
      gap: 10px;
      margin-top: 22px;
      padding-top: 16px;
      border-top: 1px solid rgba(23, 33, 25, 0.06);
    }

    .blank-card-dot {
      width: 10px;
      height: 10px;
      border-radius: 50%;
      background: var(--accent-bright);
      box-shadow: 0 0 0 7px rgba(45, 138, 112, 0.12);
      flex-shrink: 0;
    }

    .blank-card-footer .blank-line {
      margin-top: 0;
    }

    .btn,
    .btn-ghost,
    .chat-send {
      min-height: 48px;
      padding: 0 18px;
      border-radius: 999px;
      border: 1px solid transparent;
      font-size: 0.92rem;
      font-weight: 700;
      transition: transform 0.2s ease, box-shadow 0.2s ease, border-color 0.2s ease, background-color 0.2s ease;
    }

    .btn {
      color: #FFFFFF;
      background: var(--accent);
      box-shadow: var(--shadow-sm);
    }

    .btn:hover,
    .chat-send:hover {
      transform: translateY(-2px);
      background: var(--accent-dark);
    }

    .btn-ghost {
      color: var(--muted);
      background: var(--ghost-bg);
      border-color: var(--border);
      backdrop-filter: blur(10px);
    }

    .btn-ghost:hover {
      transform: translateY(-2px);
      color: var(--accent);
      background: var(--ghost-hover-bg);
      border-color: rgba(33, 94, 70, 0.24);
    }

    .desk {
      margin-top: -18px;
      padding: 2rem;
    }

    .desk-grid {
      display: grid;
      grid-template-columns: 270px minmax(0, 1fr);
      gap: 1.5rem;
      align-items: center;
    }

    .desk-label {
      margin: 0;
      color: var(--accent);
      font-size: 0.72rem;
      font-weight: 800;
      letter-spacing: 0.16em;
      text-transform: uppercase;
    }

    .desk-title {
      margin: 8px 0 0;
      font-size: 1.45rem;
      font-weight: 800;
      letter-spacing: -0.03em;
    }

    .desk-copy {
      margin: 8px 0 0;
      color: var(--muted);
      font-size: 0.88rem;
      line-height: 1.72;
    }

    .control-grid {
      display: grid;
      grid-template-columns: repeat(3, minmax(0, 1fr)) auto auto;
      gap: 12px;
      align-items: center;
    }

    .control,
    .chat-input {
      width: 100%;
      min-height: 48px;
      padding: 0 16px;
      border-radius: 1rem;
      border: 1px solid var(--border);
      background: var(--surface);
      color: var(--text);
      outline: none;
      box-shadow: var(--shadow-sm);
    }

    .control::placeholder,
    .chat-input::placeholder {
      color: var(--muted-soft);
    }

    .control:focus,
    .chat-input:focus {
      border-color: rgba(33, 94, 70, 0.42);
      box-shadow: 0 0 0 4px rgba(33, 94, 70, 0.08);
    }

    .status-line {
      min-height: 18px;
      margin-top: 12px;
      color: var(--muted);
      font-size: 0.78rem;
      text-align: right;
    }

    .status-line.is-error {
      color: var(--warm-status);
    }

    .stats {
      display: grid;
      grid-template-columns: repeat(4, minmax(0, 1fr));
      gap: 14px;
      margin-top: 20px;
    }

    .stat {
      padding: 2rem;
    }

    .stat-head {
      display: flex;
      align-items: center;
      gap: 10px;
      color: var(--muted);
      font-size: 0.74rem;
      font-weight: 600;
    }

    .stat-dot {
      width: 10px;
      height: 10px;
      border-radius: 50%;
      background: var(--accent-soft);
      border: 2px solid var(--accent);
      flex-shrink: 0;
    }

    .stat-value {
      margin-top: 14px;
      font-size: 1.6rem;
      font-weight: 800;
      letter-spacing: -0.04em;
    }

    .stat-copy {
      margin-top: 4px;
      color: var(--muted);
      font-size: 0.78rem;
    }

    .ticker {
      display: flex;
      align-items: center;
      gap: 14px;
      min-height: 48px;
      margin-top: 20px;
      padding: 0 18px;
      overflow: hidden;
      border-radius: 999px;
    }

    .ticker-badge {
      flex-shrink: 0;
      display: inline-flex;
      align-items: center;
      min-height: 30px;
      padding: 0 12px;
      border-radius: 999px;
      background: var(--ticker-badge-bg);
      color: var(--ticker-badge-text);
      font-size: 0.66rem;
      font-weight: 800;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .ticker-window {
      min-width: 0;
      overflow: hidden;
      white-space: nowrap;
      color: var(--muted);
      font-size: 0.84rem;
    }

    .ticker-track {
      display: inline-flex;
      align-items: center;
      gap: 28px;
      animation: tickerMove 92s linear infinite;
    }

    .ticker:hover .ticker-track {
      animation-play-state: paused;
    }

    .ticker-item {
      display: inline-flex;
      align-items: center;
      gap: 8px;
      white-space: nowrap;
    }

    .ticker-item strong {
      color: var(--text);
      font-weight: 700;
    }

    .brief {
      display: none;
      margin-top: 20px;
      padding: 2.5rem;
    }

    .brief.is-visible {
      display: block;
    }

    .brief-top {
      display: flex;
      align-items: center;
      gap: 12px;
      margin-bottom: 12px;
    }

    .brief-label {
      color: var(--accent);
      font-size: 0.68rem;
      font-weight: 800;
      letter-spacing: 0.16em;
      text-transform: uppercase;
    }

    .brief-rule {
      flex: 1;
      height: 1px;
      background: var(--border);
    }

    .brief-text {
      margin: 0;
      font-size: 0.96rem;
      line-height: 1.8;
      color: var(--text);
    }

    .brief-topics {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
      margin-top: 14px;
    }

    .brief-topic,
    .meta-pill,
    .trending-pill,
    .lead-fact {
      display: inline-flex;
      align-items: center;
      min-height: 32px;
      padding: 0 12px;
      border-radius: 999px;
      border: 1px solid var(--border);
      background: var(--surface-soft);
      color: var(--muted);
      font-size: 0.75rem;
      font-weight: 600;
    }

    .lead-fact {
      background: rgba(255, 253, 249, 0.92);
      color: var(--text);
      font-weight: 700;
    }

    body[data-theme="dark"] .surface,
    body[data-theme="dark"] .insight-card,
    body[data-theme="dark"] .loading-lead,
    body[data-theme="dark"] .quiet-panel {
      background: #1E1E1E;
    }

    body[data-theme="dark"] .insight-card,
    body[data-theme="dark"] .quiet-panel,
    body[data-theme="dark"] .loading-lead {
      border-color: rgba(255, 255, 255, 0.08);
    }

    body[data-theme="dark"] .insight-card--wordcloud .visual-frame,
    body[data-theme="dark"] .wordcloud-frame,
    body[data-theme="dark"] .wordcloud-stage {
      background: transparent;
    }

    body[data-theme="dark"] .lead-headline,
    body[data-theme="dark"] .lead-headline a,
    body[data-theme="dark"] .rail-headline,
    body[data-theme="dark"] .rail-headline a,
    body[data-theme="dark"] .coverage-headline,
    body[data-theme="dark"] .coverage-headline a,
    body[data-theme="dark"] .section-title,
    body[data-theme="dark"] .desk-title,
    body[data-theme="dark"] .quiet-title,
    body[data-theme="dark"] .quiet-title-small {
      color: #FFFFFF;
    }

    body[data-theme="dark"] .lead-summary,
    body[data-theme="dark"] .rail-summary,
    body[data-theme="dark"] .coverage-summary,
    body[data-theme="dark"] .coverage-meta,
    body[data-theme="dark"] .rail-meta,
    body[data-theme="dark"] .lead-note,
    body[data-theme="dark"] .quiet-copy,
    body[data-theme="dark"] .stat-copy,
    body[data-theme="dark"] .desk-copy {
      color: #B0B0B0;
    }

    body[data-theme="dark"] .brief-topic,
    body[data-theme="dark"] .meta-pill,
    body[data-theme="dark"] .trending-pill,
    body[data-theme="dark"] .lead-fact,
    body[data-theme="dark"] .sentiment {
      background: rgba(255, 255, 255, 0.10);
      color: #FFFFFF;
      border-color: rgba(255, 255, 255, 0.12);
    }

    body[data-theme="dark"] .sentiment.positive,
    body[data-theme="dark"] .sentiment.negative,
    body[data-theme="dark"] .sentiment.neutral {
      color: #FFFFFF;
    }

    body[data-theme="dark"] .rail-item:hover {
      background: #232323;
    }

    .section {
      margin-top: 20px;
    }

    .main-grid {
      display: grid;
      grid-template-columns: repeat(12, minmax(0, 1fr));
      gap: 20px;
      align-items: start;
    }

    .lead-card {
      grid-column: span 8;
      padding: 2.5rem;
    }

    .side-column {
      grid-column: span 4;
      display: grid;
      gap: 1rem;
    }

    .news-card,
    .story-card,
    .insight-card,
    .log-card {
      animation: cardRise 0.55s cubic-bezier(0.2, 0.8, 0.2, 1) both;
    }

    .lead-story-layout {
      display: block;
    }

    .lead-story-copy {
      min-width: 0;
    }

    .lead-kicker {
      display: flex;
      align-items: center;
      justify-content: flex-start;
      gap: 12px;
      flex-wrap: wrap;
      padding-bottom: 16px;
      border-bottom: 1px solid rgba(23, 33, 25, 0.08);
    }

    .lead-kicker-left,
    .lead-kicker-right {
      display: flex;
      align-items: center;
      flex-wrap: wrap;
      gap: 8px;
      justify-content: flex-start;
    }

    .lead-body {
      text-align: left;
    }

    .lead-overline {
      margin: 20px 0 0;
      display: flex;
      align-items: center;
      justify-content: flex-start;
      gap: 10px;
      color: var(--accent);
      font-size: 0.72rem;
      font-weight: 800;
      letter-spacing: 0.14em;
      text-transform: uppercase;
    }

    .lead-headline {
      margin: 18px 0 0;
      max-width: 18ch;
      font-family: "Newsreader", Georgia, serif;
      font-size: clamp(2.5rem, 4.4vw, 3.35rem);
      font-weight: 700;
      line-height: 1.2;
      letter-spacing: -0.02em;
    }

    .lead-headline a:hover,
    .rail-headline a:hover,
    .coverage-headline a:hover,
    .chat-article-title a:hover {
      color: var(--accent);
    }

    .lead-summary {
      margin: 18px 0 0;
      color: var(--muted);
      font-size: 1rem;
      line-height: 1.82;
      max-width: 66ch;
    }

    .lead-facts {
      display: flex;
      flex-wrap: wrap;
      justify-content: flex-start;
      gap: 10px;
      margin-top: 20px;
    }

    .lead-note {
      max-width: 62ch;
      margin: 20px 0 0;
      padding-top: 18px;
      border-top: 1px solid var(--border);
      color: var(--muted);
      font-size: 0.86rem;
      line-height: 1.75;
    }

    .sentiment {
      display: inline-flex;
      align-items: center;
      min-height: 24px;
      padding: 0 10px;
      border-radius: 999px;
      border: 1px solid transparent;
      font-size: 0.68rem;
      font-weight: 700;
    }

    .sentiment.positive {
      background: var(--positive-bg);
      color: var(--positive-text);
      border-color: var(--positive-border);
    }

    .sentiment.negative {
      background: var(--negative-bg);
      color: var(--negative-text);
      border-color: var(--negative-border);
    }

    .sentiment.neutral {
      background: var(--neutral-bg);
      color: var(--neutral-text);
      border-color: var(--neutral-border);
    }

    .rail {
      padding: 2.5rem;
    }

    .rail-item {
      display: block;
      padding: 1.05rem 0;
      border-bottom: 1px solid var(--border);
      transition: transform 0.22s ease, background-color 0.22s ease, box-shadow 0.22s ease;
      border-radius: 1rem;
    }

    .rail-item:last-child {
      border-bottom: 0;
    }

    .rail-item:hover {
      transform: translateY(-4px);
      background: color-mix(in srgb, var(--surface) 75%, var(--surface-soft) 25%);
      box-shadow: var(--shadow-sm);
    }

    .rail-topline {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 12px;
      flex-wrap: wrap;
    }

    .signal-dot {
      width: 8px;
      height: 8px;
      border-radius: 50%;
      background: var(--accent);
      box-shadow: 0 0 0 6px rgba(33, 94, 70, 0.14);
      animation: pulse 2.2s ease-in-out infinite;
    }

    .rail-meta {
      display: flex;
      align-items: center;
      flex-wrap: wrap;
      gap: 8px;
      color: var(--muted);
      font-size: 0.78rem;
    }

    .rail-source,
    .coverage-source {
      color: var(--text);
      font-weight: 700;
    }

    .rail-body {
      min-width: 0;
      display: grid;
      gap: 8px;
    }

    .meta-dot {
      display: inline-block;
      width: 4px;
      height: 4px;
      border-radius: 50%;
      background: var(--border);
      flex-shrink: 0;
    }

    .rail-headline {
      margin: 0;
      font-size: 1.04rem;
      font-weight: 700;
      line-height: 1.45;
      letter-spacing: -0.02em;
    }

    .rail-summary {
      color: var(--muted);
      font-size: 0.84rem;
      line-height: 1.68;
    }

    .trending {
      display: none;
      padding: 2.5rem;
    }

    .trending.is-visible {
      display: block;
    }

    .trending-list {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
    }

    .coverage {
      padding: 2.5rem;
    }

    .section-head {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 14px;
      margin-bottom: 18px;
      flex-wrap: wrap;
    }

    .section-title {
      margin: 0;
      font-size: 1.3rem;
      font-weight: 800;
      letter-spacing: -0.04em;
    }

    .count-badge {
      display: none !important;
      align-items: center;
      min-height: 34px;
      padding: 0 12px;
      border-radius: 999px;
      border: 1px solid var(--border);
      background: var(--surface-soft);
      color: var(--muted);
      font-size: 0.76rem;
      font-weight: 700;
    }

    .count-badge.is-visible {
      display: inline-flex;
    }

    .coverage-grid {
      display: grid;
      grid-template-columns: repeat(3, minmax(0, 1fr));
      gap: 16px;
    }

    .coverage-card {
      padding: 2.5rem;
      min-height: 250px;
    }

    .coverage-meta {
      display: flex;
      align-items: center;
      flex-wrap: wrap;
      gap: 8px;
      color: var(--muted);
      font-size: 0.78rem;
    }

    .coverage-headline {
      margin: 14px 0 0;
      font-family: "Newsreader", Georgia, serif;
      font-size: 1.22rem;
      font-weight: 700;
      line-height: 1.35;
      letter-spacing: -0.02em;
    }

    .coverage-summary {
      margin-top: 10px;
      color: var(--muted);
      font-size: 0.86rem;
      line-height: 1.72;
    }

    .coverage-link {
      display: inline-flex;
      align-items: center;
      margin-top: 12px;
      color: var(--accent);
      font-size: 0.82rem;
      font-weight: 700;
    }

    .insights {
      padding: 2.5rem;
    }

    #results-container {
      display: none;
    }

    .insights-grid {
      display: grid;
      grid-template-columns: repeat(12, minmax(0, 1fr));
      grid-auto-rows: minmax(200px, auto);
      gap: 1rem;
    }

    .insight-card {
      padding: 1.5rem;
      min-height: 450px;
      display: flex;
      flex-direction: column;
      gap: 0.75rem;
      overflow: hidden;
      border-radius: 2rem;
      border: 1px solid var(--border);
      background: var(--card-bg);
      background-image: none;
      box-shadow: 0 18px 42px rgba(23, 33, 25, 0.06);
    }

    .insight-card--wordcloud {
      grid-column: span 7;
      grid-row: span 2;
    }

    .insight-card--pie,
    .insight-card--bar {
      grid-column: span 5;
    }

    .insight-card--pie {
      min-height: 500px;
    }

    .insight-card--scatter {
      grid-column: span 12;
      background: var(--card-bg) !important;
    }

    .insight-head {
      margin-bottom: 0;
      flex-shrink: 0;
    }

    .insight-label {
      display: flex;
      align-items: center;
      gap: 8px;
      margin: 0;
      color: var(--muted);
      font-size: 0.66rem;
      font-weight: 700;
      letter-spacing: 0.14em;
      text-transform: uppercase;
    }

    .insight-label::before {
      content: "";
      width: 8px;
      height: 8px;
      border-radius: 50%;
      background: var(--accent);
      flex-shrink: 0;
    }

    .visual-frame {
      flex: 1 1 auto;
      display: flex;
      flex-direction: column;
      width: 100%;
      height: 100%;
      min-height: 390px;
      position: relative;
      padding: 0;
      border-radius: 1.5rem;
      background-color: transparent;
      background-image: none;
      border: 0;
      overflow: hidden;
      box-shadow: none;
    }

    .insight-card--wordcloud .visual-frame {
      min-height: 0;
      background: transparent !important;
    }

    .insight-card--pie .visual-frame {
      min-height: 440px;
      overflow: visible;
    }

    .insight-card--bar .visual-frame {
      animation: barGlow 3.6s ease-in-out infinite;
      box-shadow: 0 0 0 0 rgba(45, 138, 112, 0.12);
    }

    .insight-card--scatter .visual-frame {
      background: transparent !important;
      border-color: transparent !important;
      box-shadow: none !important;
      padding: 0;
      border-radius: 0 !important;
    }

    body[data-theme="dark"] .visual-frame,
    body[data-theme="dark"] .scatter-frame {
      background: transparent !important;
      background-image: none !important;
      border-color: transparent !important;
    }

    .scatter-frame {
      flex: 1 1 auto;
      min-height: 450px;
      height: 100%;
      width: 100%;
      position: relative;
      overflow: hidden;
      border: 0 !important;
      border-radius: 0;
      background: transparent !important;
      background-image: none !important;
      box-shadow: none !important;
      padding: 0;
    }

    .insight-card--scatter #scatter-chart,
    .insight-card--scatter #scatter-chart.visual-frame,
    .insight-card--scatter .visual-frame#scatter-chart {
      background: transparent !important;
      background-image: none !important;
      border: 0 !important;
      border-radius: 0 !important;
      box-shadow: none !important;
      padding: 0 !important;
    }

    #pie-chart,
    #bar-chart,
    #scatter-chart {
      min-height: 390px;
      width: 100%;
      position: relative;
      overflow: hidden;
    }

    #scatter-chart,
    #scatter-chart .js-plotly-plot,
    #scatter-chart .plot-container,
    #scatter-chart .svg-container,
    #scatter-chart .main-svg,
    #scatter-chart .bg {
      background: transparent !important;
      background-image: none !important;
      box-shadow: none !important;
      border: 0 !important;
    }

    #scatter-chart {
      min-height: 450px;
      height: 450px;
    }

    #scatter-chart .bg,
    #scatter-chart .plotbg,
    #scatter-chart .bglayer rect,
    #scatter-chart .cartesianlayer .subplot rect.bg {
      fill: transparent !important;
      stroke: transparent !important;
    }

    #pie-chart {
      min-height: 440px;
      overflow: visible;
    }

    .wordcloud-frame {
      flex: 1 1 auto;
      min-height: 0;
      display: flex;
      align-items: stretch;
      background: transparent !important;
    }

    .wordcloud-stage {
      flex: 1 1 auto;
      width: 100%;
      min-height: 100%;
      height: 100%;
      border-radius: 1.25rem;
      border: 0;
      background-color: transparent !important;
      display: flex;
      align-items: center;
      justify-content: center;
      overflow: hidden;
      padding: 0;
    }

    .wordcloud-image {
      display: block;
      width: 100%;
      height: 100%;
      max-height: 100%;
      object-fit: contain;
      margin: 0 auto;
    }

    .wordcloud-stage.is-loading {
      position: relative;
      overflow: hidden;
      background:
        linear-gradient(135deg, rgba(33, 94, 70, 0.05), rgba(98, 111, 125, 0.08)),
        var(--surface-soft);
    }

    .wordcloud-stage.is-loading::after {
      content: "";
      position: absolute;
      inset: 0;
      background: linear-gradient(90deg, transparent, rgba(255, 255, 255, 0.86), transparent);
      transform: translateX(-100%);
      animation: shimmer 1.25s linear infinite;
    }

    .visual-empty {
      width: 100%;
      min-height: 100%;
      height: 100%;
      border-radius: 1.25rem;
      background: transparent;
    }

    .agent-log {
      display: none;
      margin-top: 22px;
    }

    .agent-log.is-visible {
      display: block;
    }

    .log-toggle {
      width: 100%;
      min-height: 56px;
      padding: 0 18px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 12px;
      border-radius: 1.25rem;
      border: 1px solid var(--border);
      background: var(--surface);
      color: var(--muted);
      font-size: 0.88rem;
      font-weight: 700;
      text-align: left;
      box-shadow: var(--shadow-sm);
    }

    .log-chevron {
      color: var(--accent);
      transition: transform 0.2s ease;
    }

    .agent-log.is-open .log-chevron {
      transform: rotate(180deg);
    }

    .log-content {
      display: none;
      margin-top: 14px;
      gap: 14px;
    }

    .agent-log.is-open .log-content {
      display: grid;
    }

    .log-card {
      padding: 2.5rem;
    }

    .log-topic {
      margin: 0 0 12px;
      font-size: 1rem;
      font-weight: 800;
      letter-spacing: -0.02em;
    }

    .log-entry + .log-entry {
      margin-top: 12px;
      padding-top: 12px;
      border-top: 1px solid var(--border);
    }

    .log-stage {
      display: block;
      margin-bottom: 5px;
      color: var(--accent);
      font-size: 0.62rem;
      font-weight: 800;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .log-message {
      color: var(--text);
      font-size: 0.84rem;
      line-height: 1.72;
    }

    .quiet-card {
      padding: 2.5rem;
    }

    .quiet-panel {
      border-radius: 1.5rem;
      border: 1px solid var(--border);
      background:
        radial-gradient(circle at top left, rgba(33, 94, 70, 0.10), transparent 28%),
        linear-gradient(180deg, #FFFFFF 0%, #F7F7F5 100%);
      padding: 2rem;
      min-height: 320px;
      display: flex;
      align-items: flex-end;
    }

    .quiet-kicker {
      display: inline-flex;
      align-items: center;
      min-height: 30px;
      padding: 0 11px;
      border-radius: 999px;
      background: #FFFFFF;
      border: 1px solid var(--border);
      color: var(--accent);
      font-size: 0.7rem;
      font-weight: 800;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .quiet-title {
      margin: 12px 0 0;
      font-family: "Newsreader", Georgia, serif;
      font-size: 1.42rem;
      font-weight: 700;
      line-height: 1.28;
      letter-spacing: -0.03em;
    }

    .quiet-copy {
      margin: 8px 0 0;
      color: var(--muted);
      font-size: 0.88rem;
      line-height: 1.72;
    }

    .quiet-rail {
      display: grid;
      gap: 12px;
      padding: 2rem;
    }

    .quiet-rail-row {
      padding: 16px 0;
      border-bottom: 1px solid var(--border);
    }

    .quiet-rail-row:last-child {
      border-bottom: 0;
    }

    .quiet-label {
      color: var(--accent);
      font-size: 0.68rem;
      font-weight: 700;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .quiet-title-small {
      margin-top: 5px;
      font-size: 0.94rem;
      font-weight: 700;
      line-height: 1.5;
    }

    .quiet-muted {
      margin-top: 6px;
      color: var(--muted);
      font-size: 0.82rem;
      line-height: 1.65;
    }

    .skeleton-box,
    .skeleton-line,
    .skeleton-pill {
      position: relative;
      overflow: hidden;
      background: #E2E8E1;
    }

    .skeleton-box::after,
    .skeleton-line::after,
    .skeleton-pill::after {
      content: "";
      position: absolute;
      inset: 0;
      background: linear-gradient(90deg, transparent, rgba(255, 255, 255, 0.74), transparent);
      transform: translateX(-100%);
      animation: shimmer 1.2s linear infinite;
    }

    .loading-lead {
      padding: 2.5rem;
      border-radius: 1.5rem;
      border: 1px solid var(--border);
      background: rgba(255, 255, 255, 0.88);
    }

    .loading-lead-copy {
      min-width: 0;
    }

    .skeleton-meta {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
      margin-top: 16px;
      justify-content: flex-start;
    }

    .skeleton-pill {
      height: 28px;
      border-radius: 999px;
    }

    .skeleton-pill.wide {
      width: 112px;
    }

    .skeleton-pill.mid {
      width: 80px;
    }

    .skeleton-line {
      height: 12px;
      margin-top: 10px;
      border-radius: 999px;
    }

    .skeleton-line.lg {
      width: 90%;
      height: 20px;
    }

    .skeleton-line.xl {
      width: 92%;
      height: 26px;
    }

    .skeleton-line.md {
      width: 76%;
    }

    .skeleton-line.sm {
      width: 52%;
    }

    .loading-rail {
      padding: 2.5rem;
    }

    .loading-rail-row {
      padding: 18px 0;
      border-bottom: 1px solid var(--border);
    }

    .loading-rail-row:last-child {
      border-bottom: 0;
    }

    .loading-coverage {
      padding: 2.5rem;
      min-height: 220px;
    }

    .chat-launch {
      position: fixed;
      right: 22px;
      bottom: 22px;
      z-index: 9999;
      display: inline-flex;
      align-items: center;
      gap: 12px;
      min-height: 58px;
      padding: 0 16px 0 14px;
      border: 1px solid rgba(23, 33, 25, 0.08);
      border-radius: 999px;
      background: rgba(255, 255, 255, 0.94);
      color: var(--text);
      box-shadow: 0 18px 38px rgba(23, 33, 25, 0.12);
    }

    .chat-launch-mark {
      width: 30px;
      height: 30px;
      display: inline-flex;
      align-items: center;
      justify-content: center;
      border-radius: 10px;
      background: rgba(45, 138, 112, 0.10);
      color: var(--accent);
      flex-shrink: 0;
    }

    .chat-launch-mark svg,
    .chat-bot-mark svg {
      width: 16px;
      height: 16px;
      stroke: currentColor;
      fill: none;
      stroke-width: 2;
      stroke-linecap: round;
      stroke-linejoin: round;
    }

    .chat-launch-copy {
      display: flex;
      flex-direction: column;
      align-items: flex-start;
      line-height: 1.1;
    }

    .chat-launch-label {
      color: var(--muted);
      font-size: 0.68rem;
      font-weight: 700;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    .chat-launch-title {
      margin-top: 4px;
      color: var(--text);
      font-size: 0.92rem;
      font-weight: 700;
    }

    .chat-launch-status {
      width: 10px;
      height: 10px;
      border-radius: 50%;
      background: var(--accent-bright);
      box-shadow: 0 0 0 5px rgba(45, 138, 112, 0.16);
      flex-shrink: 0;
      animation: pulse 3s ease-in-out infinite;
    }

    .chat-overlay {
      position: fixed;
      inset: 0;
      background: rgba(26, 26, 26, 0.16);
      opacity: 0;
      pointer-events: none;
      z-index: 9997;
      transition: opacity 0.22s ease;
    }

    .chat-overlay.is-visible {
      opacity: 1;
      pointer-events: auto;
    }

    .chat-panel {
      position: fixed;
      right: 22px;
      bottom: 92px;
      width: 390px;
      max-width: calc(100vw - 28px);
      height: min(720px, calc(100vh - 120px));
      display: flex;
      flex-direction: column;
      overflow: hidden;
      border-radius: 1.75rem;
      border: 1px solid rgba(229, 231, 235, 0.95);
      background: rgba(255, 255, 255, 0.96);
      backdrop-filter: blur(18px);
      box-shadow: var(--shadow-lg);
      transform: translateY(18px) scale(0.98);
      opacity: 0;
      pointer-events: none;
      z-index: 9998;
      transition: transform 0.22s ease, opacity 0.22s ease;
    }

    .chat-panel.is-open {
      transform: translateY(0) scale(1);
      opacity: 1;
      pointer-events: auto;
    }

    .chat-header {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 12px;
      padding: 18px;
      border-bottom: 1px solid rgba(229, 231, 235, 0.9);
      background: linear-gradient(180deg, rgba(255, 255, 255, 0.98), rgba(248, 247, 244, 0.92));
    }

    .chat-brand {
      display: flex;
      align-items: center;
      gap: 12px;
      min-width: 0;
    }

    .chat-bot-mark {
      width: 40px;
      height: 40px;
      display: inline-flex;
      align-items: center;
      justify-content: center;
      border-radius: 14px;
      background: linear-gradient(145deg, #1A563F 0%, #2C7B5D 100%);
      color: #FFFFFF;
      flex-shrink: 0;
    }

    .chat-brand-title {
      font-size: 0.96rem;
      font-weight: 800;
    }

    .chat-brand-sub {
      margin-top: 3px;
      color: var(--muted);
      font-size: 0.76rem;
    }

    .chat-close {
      width: 38px;
      height: 38px;
      border: 1px solid var(--border);
      border-radius: 12px;
      background: var(--surface);
      color: var(--muted);
      font-size: 1rem;
      font-weight: 700;
      box-shadow: var(--shadow-sm);
    }

    .chat-close:hover {
      background: #FFFFFF;
      color: var(--text);
    }

    .chat-subhead {
      padding: 10px 18px 12px;
      color: var(--muted);
      font-size: 0.76rem;
      border-bottom: 1px solid rgba(229, 231, 235, 0.7);
      background: rgba(255, 255, 255, 0.84);
    }

    .chat-messages {
      flex: 1;
      overflow-y: auto;
      display: flex;
      flex-direction: column;
      gap: 12px;
      padding: 16px 18px;
      scrollbar-width: thin;
      scrollbar-color: rgba(26, 26, 26, 0.18) transparent;
    }

    .chat-bubble {
      max-width: 100%;
      font-size: 0.86rem;
      line-height: 1.65;
    }

    .chat-bubble.user {
      align-self: flex-end;
      max-width: 84%;
      padding: 12px 14px;
      border-radius: 20px 20px 8px 20px;
      background: #1A1A1A;
      color: #FFFFFF;
      box-shadow: var(--shadow-sm);
    }

    .chat-bubble.bot {
      align-self: flex-start;
      width: 100%;
      padding: 12px 14px;
      border-radius: 20px 20px 20px 8px;
      background: var(--surface-soft);
      border: 1px solid var(--border);
      color: var(--text);
    }

    .chat-bubble p {
      margin: 0;
    }

    .chat-articles {
      display: grid;
      gap: 8px;
      margin-top: 10px;
    }

    .chat-article {
      padding: 10px 12px;
      border-radius: 1rem;
      border: 1px solid var(--border);
      border-left: 3px solid var(--accent);
      background: #FFFFFF;
    }

    .chat-article.positive {
      border-left-color: var(--positive-text);
    }

    .chat-article.negative {
      border-left-color: var(--negative-text);
    }

    .chat-article-title {
      margin: 0;
      font-size: 0.84rem;
      font-weight: 700;
      line-height: 1.45;
    }

    .chat-article-meta,
    .chat-commentary {
      margin-top: 4px;
      color: var(--muted);
      font-size: 0.74rem;
      line-height: 1.55;
    }

    .chat-suggestions {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
      padding: 10px 18px 0;
    }

    .chat-chip,
    .quick-briefing {
      min-height: 34px;
      padding: 0 12px;
      border-radius: 999px;
      border: 1px solid var(--border);
      background: var(--surface-soft);
      color: var(--muted);
      font-size: 0.76rem;
      font-weight: 600;
      transition: transform 0.2s ease, border-color 0.2s ease, color 0.2s ease, background-color 0.2s ease;
    }

    .chat-chip:hover,
    .quick-briefing:hover {
      transform: translateY(-1px);
      border-color: rgba(33, 94, 70, 0.24);
      color: var(--accent);
      background: #FFFFFF;
    }

    .quick-briefing {
      margin: 12px 18px 0;
      border-color: rgba(45, 138, 112, 0.28);
      background: #2D8A70;
      color: #FFFFFF;
      box-shadow: 0 12px 26px rgba(45, 138, 112, 0.18);
    }

    .chat-input-row {
      display: flex;
      gap: 10px;
      padding: 16px 18px 18px;
      border-top: 1px solid rgba(229, 231, 235, 0.75);
      background: linear-gradient(180deg, rgba(255, 255, 255, 0.78), rgba(255, 255, 255, 0.96));
    }

    .chat-send {
      min-width: 84px;
      color: #FFFFFF;
      background: #2D8A70;
    }

    .quick-briefing:hover,
    .chat-send:hover {
      background: #246F5A;
      color: #FFFFFF;
    }

    body[data-theme="dark"] .chat-launch {
      border: 1px solid rgba(255, 255, 255, 0.10);
      background: #1E1E1E;
      color: #F5F5F5;
      box-shadow: 0 18px 38px rgba(0, 0, 0, 0.40);
    }

    body[data-theme="dark"] .chat-launch-label {
      color: #B0B0B0;
    }

    body[data-theme="dark"] .chat-launch-title {
      color: #FFFFFF;
    }

    body[data-theme="dark"] .chat-launch-mark {
      background: rgba(45, 138, 112, 0.18);
      color: #DDF7EE;
    }

    body[data-theme="dark"] .chat-launch-status {
      background: #2D8A70;
      box-shadow: 0 0 0 5px rgba(45, 138, 112, 0.20);
    }

    body[data-theme="dark"] .chat-overlay {
      background: rgba(0, 0, 0, 0.46);
    }

    body[data-theme="dark"] .chat-panel {
      border: 1px solid rgba(255, 255, 255, 0.10);
      background: #1E1E1E;
      backdrop-filter: none;
      box-shadow: 0 32px 80px rgba(0, 0, 0, 0.48);
    }

    body[data-theme="dark"] .chat-header {
      border-bottom: 1px solid rgba(255, 255, 255, 0.10);
      background: #1E1E1E;
    }

    body[data-theme="dark"] .chat-brand-title,
    body[data-theme="dark"] .chat-bubble.bot,
    body[data-theme="dark"] .chat-bubble.bot p,
    body[data-theme="dark"] .chat-bubble.user,
    body[data-theme="dark"] .chat-article-title,
    body[data-theme="dark"] .chat-article-title a {
      color: #FFFFFF;
    }

    body[data-theme="dark"] .chat-brand-sub,
    body[data-theme="dark"] .chat-subhead,
    body[data-theme="dark"] .chat-article-meta,
    body[data-theme="dark"] .chat-commentary {
      color: #B0B0B0;
    }

    body[data-theme="dark"] .chat-close {
      border: 1px solid rgba(255, 255, 255, 0.12);
      background: #121212;
      color: #F5F5F5;
      box-shadow: none;
    }

    body[data-theme="dark"] .chat-close:hover {
      background: #181818;
      color: #FFFFFF;
    }

    body[data-theme="dark"] .chat-subhead {
      border-bottom: 1px solid rgba(255, 255, 255, 0.08);
      background: #1E1E1E;
    }

    body[data-theme="dark"] .chat-messages {
      background: #1E1E1E;
      scrollbar-color: rgba(255, 255, 255, 0.16) transparent;
    }

    body[data-theme="dark"] .chat-bubble.user {
      background: #2D8A70;
      box-shadow: none;
    }

    body[data-theme="dark"] .chat-bubble.bot {
      background: #232323;
      border: 1px solid rgba(255, 255, 255, 0.08);
    }

    body[data-theme="dark"] .chat-article {
      background: #121212;
      border: 1px solid rgba(255, 255, 255, 0.08);
    }

    body[data-theme="dark"] .chat-input-row {
      border-top: 1px solid rgba(255, 255, 255, 0.08);
      background: #1E1E1E;
    }

    body[data-theme="dark"] .chat-input {
      background: #121212;
      border: 1px solid rgba(255, 255, 255, 0.14);
      color: #FFFFFF;
      box-shadow: none;
    }

    body[data-theme="dark"] .chat-input::placeholder {
      color: #8F8F8F;
    }

    body[data-theme="dark"] .chat-input:focus {
      border-color: rgba(45, 138, 112, 0.56);
      box-shadow: 0 0 0 4px rgba(45, 138, 112, 0.12);
    }

    body[data-theme="dark"] .quick-briefing,
    body[data-theme="dark"] .chat-send {
      border-color: transparent;
      background: #2D8A70;
      color: #FFFFFF;
      box-shadow: none;
    }

    body[data-theme="dark"] .quick-briefing:hover,
    body[data-theme="dark"] .chat-send:hover {
      background: #246F5A;
      color: #FFFFFF;
    }

    .is-loading {
      opacity: 0.65;
      pointer-events: none;
    }

    .visually-hidden {
      position: absolute;
      width: 1px;
      height: 1px;
      padding: 0;
      margin: -1px;
      overflow: hidden;
      clip: rect(0, 0, 0, 0);
      white-space: nowrap;
      border: 0;
    }

    @keyframes tickerMove {
      from { transform: translateX(0); }
      to { transform: translateX(-50%); }
    }

    @keyframes synthesisPulse {
      0% { transform: translateX(-120%); }
      55% { transform: translateX(100%); }
      100% { transform: translateX(220%); }
    }

    @keyframes shimmer {
      from { transform: translateX(-100%); }
      to { transform: translateX(100%); }
    }

    @keyframes pulse {
      0%, 100% { transform: scale(1); opacity: 1; }
      50% { transform: scale(0.9); opacity: 0.58; }
    }

    @keyframes trackingInExpand {
      0% {
        opacity: 0;
        letter-spacing: 0.08em;
        transform: translateY(10px);
      }
      100% {
        opacity: 1;
        letter-spacing: 0.28em;
        transform: translateY(0);
      }
    }

    @keyframes barGlow {
      0%, 100% {
        box-shadow: 0 0 0 0 rgba(45, 138, 112, 0.10);
      }
      50% {
        box-shadow: 0 18px 34px rgba(45, 138, 112, 0.16);
      }
    }

    @keyframes heroArrowPulse {
      0%, 100% {
        transform: translateY(0);
        box-shadow: 0 10px 24px rgba(23, 33, 25, 0.08);
      }
      50% {
        transform: translateY(6px);
        box-shadow: 0 16px 30px rgba(23, 33, 25, 0.12);
      }
    }

    @keyframes cardRise {
      from {
        opacity: 0;
        transform: translateY(16px);
      }
      to {
        opacity: 1;
        transform: translateY(0);
      }
    }

    @media (max-width: 1220px) {
      .desk-grid,
      .stats,
      .coverage-grid {
        grid-template-columns: 1fr 1fr;
      }

      .main-grid {
        grid-template-columns: 1fr;
      }

      .lead-card,
      .side-column {
        grid-column: span 1;
      }

      .control-grid {
        grid-template-columns: repeat(3, minmax(0, 1fr));
      }

      .control-grid #fetchBtn,
      .control-grid #sampleBtn {
        grid-column: span 1;
      }

      .blank-stack {
        width: min(720px, 100%);
      }
    }

    @media (max-width: 900px) {
      .shell {
        padding-left: 18px;
        padding-right: 18px;
      }

      .hero {
        padding-left: 18px;
        padding-right: 18px;
      }

      .desk-grid,
      .stats,
      .coverage-grid {
        grid-template-columns: 1fr;
      }

      .control-grid {
        grid-template-columns: 1fr 1fr;
      }

      .hero-actions .control {
        min-width: 190px;
      }

      .status-line {
        text-align: left;
      }

      .blank-stack {
        width: min(620px, 100%);
        height: 380px;
      }

      .blank-card {
        width: min(340px, calc(100% - 20px));
      }

      .hero-card-title {
        font-size: clamp(1.7rem, 4.2vw, 2.35rem);
      }

      .hero-card-stat-grid {
        grid-template-columns: 1fr;
      }

      .blank-card--one {
        --card-transform: translateX(calc(-50% + 38px)) rotate(6deg) translateY(28px);
        --card-transform-hidden: translateX(calc(-50% + 38px)) rotate(6deg) translateY(54px) scale(0.985);
        --card-transform-loaded: translateX(calc(-50% + 52px)) rotate(7deg) translateY(22px);
        --card-transform-hover: translateX(calc(-50% + 320px)) rotate(2deg) translateY(4px);
        left: 50%;
        bottom: 28px;
      }

      .blank-card--two {
        --card-transform: translateX(calc(-50% - 38px)) rotate(-6deg) translateY(28px);
        --card-transform-hidden: translateX(calc(-50% - 38px)) rotate(-6deg) translateY(54px) scale(0.985);
        --card-transform-loaded: translateX(calc(-50% - 52px)) rotate(-7deg) translateY(16px);
        --card-transform-hover: translateX(calc(-50% - 320px)) rotate(-2deg) translateY(4px);
        left: 50%;
        bottom: 28px;
      }

      .blank-card--three {
        --card-transform-loaded: translateX(-50%) translateY(-8px) scale(1);
        --card-transform-hover: translateX(-50%) translateY(-10px) scale(1);
        width: min(430px, calc(100% - 12px));
      }

      .insights-grid {
        grid-template-columns: repeat(2, minmax(0, 1fr));
      }

      .insight-card--wordcloud,
      .insight-card--scatter {
        grid-column: span 2;
      }

      .insight-card--pie,
      .insight-card--bar {
        grid-column: span 1;
      }
    }

    @media (max-width: 720px) {
      .logo-wrapper {
        font-size: 2.35rem;
      }

      .hero-visuals {
        padding-left: 0;
        padding-right: 0;
        min-height: 340px;
      }

      .hero-actions .control,
      .hero-actions .theme-toggle,
      .hero-actions .btn,
      .hero-actions .btn-ghost {
        width: 100%;
      }

      .blank-stack {
        width: min(100%, 400px);
        height: 300px;
      }

      .blank-card {
        width: min(248px, calc(100% - 10px));
        min-height: 188px;
        padding: 1rem;
        border-radius: 1.25rem;
      }

      .hero-card-shell {
        gap: 0.75rem;
      }

      .hero-card-meta,
      .hero-card-footer {
        align-items: flex-start;
        flex-direction: column;
      }

      .hero-card-title {
        font-size: 1.45rem;
      }

      .hero-card-summary,
      .hero-card-note,
      .hero-card-caption {
        font-size: 0.78rem;
        line-height: 1.58;
      }

      .hero-card-arrow {
        width: 34px;
        height: 34px;
      }

      .blank-card--one {
        --card-transform: translateX(calc(-50% + 28px)) rotate(5deg) translateY(22px);
        --card-transform-hidden: translateX(calc(-50% + 28px)) rotate(5deg) translateY(46px) scale(0.985);
        --card-transform-loaded: translateX(calc(-50% + 38px)) rotate(6deg) translateY(18px);
        --card-transform-hover: translateX(calc(-50% + 320px)) rotate(2deg) translateY(4px);
        left: 50%;
        bottom: 22px;
      }

      .blank-card--two {
        --card-transform: translateX(calc(-50% - 28px)) rotate(-5deg) translateY(22px);
        --card-transform-hidden: translateX(calc(-50% - 28px)) rotate(-5deg) translateY(46px) scale(0.985);
        --card-transform-loaded: translateX(calc(-50% - 38px)) rotate(-6deg) translateY(14px);
        --card-transform-hover: translateX(calc(-50% - 320px)) rotate(-2deg) translateY(4px);
        left: 50%;
        bottom: 22px;
      }

      .blank-card--three {
        --card-transform-loaded: translateX(-50%) translateY(-6px) scale(1);
        --card-transform-hover: translateX(-50%) translateY(-10px) scale(1);
        width: min(300px, calc(100% - 6px));
        min-height: 214px;
      }

      .blank-card-frame {
        grid-template-columns: 1fr;
      }

      .insights-grid {
        grid-template-columns: 1fr;
      }

      .insight-card--wordcloud,
      .insight-card--pie,
      .insight-card--bar,
      .insight-card--scatter {
        grid-column: span 1;
      }

      .control-grid {
        grid-template-columns: 1fr;
      }

      .chat-launch {
        right: 18px;
        bottom: 18px;
      }

      .chat-panel {
        right: 16px;
        bottom: 86px;
        width: calc(100vw - 32px);
        height: min(720px, calc(100vh - 118px));
      }
    }
  </style>
</head>
<body>
  <header class="hero">
    <div class="hero-inner">
      <div class="logo-wrapper" role="heading" aria-level="1" aria-label="VeritasAI">
        <span class="veritas-serif">Ver<span class="veritas-i">ı</span>tas</span>
        <span class="ai-sans">AI</span>
      </div>
      <p class="hero-tagline">PRECISION NEWS SYNTHESIS</p>
      <p class="hero-copy">
        VeritasAI transforms fragmented RSS streams into a singular, cohesive news experience.
        By prioritizing signal over noise, it synthesizes global coverage into an actionable live edition with integrated visual analysis.
      </p>
      <div class="hero-actions">
        <select class="control" id="languageSelect" aria-label="Language">
          <option value="English">English (Direct)</option>
          <option value="Spanish">Spanish (Neural Translation)</option>
          <option value="Hindi">Hindi (Neural Translation)</option>
          <option value="French">French (Neural Translation)</option>
          <option value="German">German (Neural Translation)</option>
        </select>
        <button class="btn" id="heroFetchBtn" type="button">Build Live Edition</button>
        <button class="theme-toggle" id="themeToggle" type="button" aria-pressed="false">Dark Mode</button>
        <button class="btn-ghost" id="heroInsightsBtn" type="button">See Insights</button>
      </div>
      <div class="hero-visuals" id="heroVisuals">
        <div class="blank-stack" id="heroBlankStack">
          <article class="blank-card blank-card--one">
            <div class="blank-card-top">
              <span class="blank-pill short"></span>
              <div class="blank-chip-row">
                <span class="blank-chip"></span>
                <span class="blank-chip"></span>
              </div>
            </div>
            <div class="blank-line xl"></div>
            <div class="blank-line lg"></div>
            <div class="blank-line md"></div>
            <div class="blank-card-frame">
              <div class="blank-card-box"></div>
              <div class="blank-card-box"></div>
            </div>
          </article>
          <article class="blank-card blank-card--two">
            <div class="blank-card-top">
              <span class="blank-pill mid"></span>
              <div class="blank-chip-row">
                <span class="blank-chip"></span>
                <span class="blank-chip"></span>
                <span class="blank-chip"></span>
              </div>
            </div>
            <div class="blank-line xl"></div>
            <div class="blank-line lg"></div>
            <div class="blank-line sm"></div>
            <div class="blank-card-footer">
              <span class="blank-card-dot"></span>
              <div class="blank-line md"></div>
            </div>
          </article>
          <article class="blank-card blank-card--three">
            <div class="blank-card-top">
              <span class="blank-pill long"></span>
              <div class="blank-chip-row">
                <span class="blank-chip"></span>
                <span class="blank-chip"></span>
              </div>
            </div>
            <div class="blank-line xl"></div>
            <div class="blank-line lg"></div>
            <div class="blank-line md"></div>
            <div class="blank-line sm"></div>
            <div class="blank-card-frame">
              <div class="blank-card-box"></div>
              <div class="blank-card-box"></div>
            </div>
            <div class="blank-card-footer">
              <span class="blank-card-dot"></span>
              <div class="blank-line lg"></div>
            </div>
          </article>
        </div>
        <div class="synthesis-progress" id="heroSynthesisProgress" hidden>
          <div class="synthesis-progress-shell">
            <p class="synthesis-progress-title">Synthesis Terminal</p>
            <div class="synthesis-progress-track" aria-hidden="true"></div>
            <p class="synthesis-progress-message" id="synthesisProgressMessage" aria-live="polite">
              Initializing Neural Engines...
            </p>
            <button class="synthesis-progress-cancel" id="cancelRequestBtn" type="button">Cancel Synthesis</button>
          </div>
        </div>
      </div>
    </div>
  </header>

  <div class="shell">
    <section class="desk surface interactive" id="desk">
      <div class="desk-grid">
        <div>
          <p class="desk-label">Editorial Desk</p>
          <h2 class="desk-title">Curate the live edition.</h2>
          <p class="desk-copy">Search by topic, switch languages on demand, and move from live coverage to reasoning and visuals without leaving the desk.</p>
        </div>
        <div>
          <div class="control-grid">
            <input class="control" id="topic1" type="text" placeholder="Topic 1" aria-label="Topic 1" />
            <input class="control" id="topic2" type="text" placeholder="Topic 2" aria-label="Topic 2" />
            <input class="control" id="topic3" type="text" placeholder="Topic 3" aria-label="Topic 3" />
            <button class="btn" id="fetchBtn" type="button">Fetch News</button>
            <button class="btn-ghost" id="sampleBtn" type="button">Load Sample Topics</button>
          </div>
          <div class="status-line" id="statusLine" aria-live="polite"></div>
        </div>
      </div>
    </section>

    <section class="stats" aria-label="Edition metrics">
      <article class="surface interactive stat">
        <div class="stat-head"><span class="stat-dot"></span>Stories surfaced</div>
        <div class="stat-value" id="statArticles">—</div>
        <div class="stat-copy">Article count in the current edition.</div>
      </article>
      <article class="surface interactive stat">
        <div class="stat-head"><span class="stat-dot"></span>Topics active</div>
        <div class="stat-value" id="statTopics">—</div>
        <div class="stat-copy">Prompt count shaping the desk.</div>
      </article>
      <article class="surface interactive stat">
        <div class="stat-head"><span class="stat-dot"></span>Positive share</div>
        <div class="stat-value" id="statPositive">—</div>
        <div class="stat-copy">Positive sentiment within fetched coverage.</div>
      </article>
      <article class="surface interactive stat">
        <div class="stat-head"><span class="stat-dot"></span>Latest timestamp</div>
        <div class="stat-value" id="statLatest">—</div>
        <div class="stat-copy">Most recent story timestamp in the edition.</div>
      </article>
    </section>

    <section class="surface ticker" aria-label="Live ticker">
      <div class="ticker-badge">Live</div>
      <div class="ticker-window">
        <div class="ticker-track" id="tickerTrack"></div>
      </div>
    </section>

    <section class="surface brief" id="briefCard" hidden>
      <div class="brief-top">
        <div class="brief-label">Briefing</div>
        <div class="brief-rule"></div>
      </div>
      <p class="brief-text" id="briefText"></p>
      <div class="brief-topics" id="briefTopics"></div>
    </section>

    <section class="section" id="editionStage">
      <div class="main-grid">
        <article class="surface interactive lead-card" id="heroArticle"></article>
        <div class="side-column">
          <section class="surface rail" id="secondaryArticles"></section>
          <section class="surface trending interactive" id="trendingSection">
            <div class="trending-list" id="trendingSources"></div>
          </section>
        </div>
      </div>
    </section>

    <div id="results-container">
      <section class="section surface coverage">
        <div class="section-head">
          <div><h2 class="section-title">More Coverage</h2></div>
          <div class="count-badge" id="coverageCount"></div>
        </div>
        <div class="coverage-grid" id="coverageGrid"></div>
      </section>

      <section class="section surface insights" id="insightsBand">
        <div class="section-head">
          <div><h2 class="section-title">Insights</h2></div>
        </div>
        <div class="insights-grid">
          <article class="surface insight-card insight-card--wordcloud interactive">
            <div class="insight-head">
              <p class="insight-label">Word Cloud</p>
            </div>
            <div class="visual-frame wordcloud-frame" id="wordcloudFrame">
              <div class="wordcloud-stage is-loading"></div>
            </div>
          </article>
          <article class="surface insight-card insight-card--pie interactive">
            <div class="insight-head">
              <p class="insight-label">Sentiment Pie</p>
            </div>
            <div class="visual-frame" id="pie-chart"></div>
          </article>
          <article class="surface insight-card insight-card--bar interactive">
            <div class="insight-head">
              <p class="insight-label">Keyword Bar</p>
            </div>
            <div class="visual-frame" id="bar-chart"></div>
          </article>
          <article class="surface insight-card insight-card--scatter interactive">
            <div class="insight-head">
              <p class="insight-label">Sentiment Scatter</p>
            </div>
            <div class="scatter-frame" id="scatter-chart"></div>
          </article>
        </div>
      </section>
    </div>

    <section class="agent-log" id="agentLogSection" hidden>
      <button class="log-toggle" id="agentLogToggle" type="button" aria-expanded="false">
        <span>Agent Reasoning Log</span>
        <span class="log-chevron">▾</span>
      </button>
      <div class="log-content" id="agentLogContent"></div>
    </section>
  </div>

  <button class="chat-launch interactive" id="chatFab" type="button" aria-label="Open VeritasBot" aria-expanded="false">
    <span class="chat-launch-mark" aria-hidden="true">
      <svg viewBox="0 0 24 24">
        <path d="M6 7h12"></path>
        <path d="M6 12h10"></path>
        <path d="M6 17h7"></path>
      </svg>
    </span>
    <span class="chat-launch-copy">
      <span class="chat-launch-label">Assistant</span>
      <span class="chat-launch-title">Ask Veritas</span>
    </span>
    <span class="chat-launch-status" aria-hidden="true"></span>
  </button>

  <div class="chat-overlay" id="chatOverlay"></div>

  <aside class="chat-panel" id="chatPanel" aria-hidden="true">
    <div class="chat-header">
      <div class="chat-brand">
        <span class="chat-bot-mark" aria-hidden="true">
          <svg viewBox="0 0 24 24">
            <path d="M6 7h12"></path>
            <path d="M6 12h10"></path>
            <path d="M6 17h7"></path>
          </svg>
        </span>
        <div>
          <div class="chat-brand-title">VeritasBot</div>
          <div class="chat-brand-sub">Ask about the live edition or request a briefing.</div>
        </div>
      </div>
      <button class="chat-close" id="chatCloseBtn" type="button" aria-label="Close chat panel">×</button>
    </div>
    <div class="chat-subhead">The assistant uses the same live article set rendered on the page.</div>
    <div class="chat-messages" id="chatMessages">
      <div class="chat-bubble bot">
        <p>Try a briefing, ask what is trending, or search for a company, market, or event.</p>
      </div>
    </div>
    <div class="chat-suggestions" id="suggestionChips"></div>
    <button class="quick-briefing" id="quickBriefingBtn" type="button">Quick Briefing</button>
    <div class="chat-input-row">
      <label class="visually-hidden" for="chatInput">Chat message</label>
      <input class="chat-input" id="chatInput" type="text" placeholder="Ask VeritasBot about the news" />
      <button class="chat-send" id="chatSendBtn" type="button">Ask</button>
    </div>
  </aside>

  <script>
    let allArticles = [];
    let originalArticles = [];
    let chatHistory = [];
    let lastTopic = null;
    let chatPanelOpen = false;

    let latestBriefing = "";
    let currentTopics = [];
    let reasoningData = {};
    let latestEditionTimestamp = null;
    let translationRequestId = 0;
    let agentLogOpen = false;
    let hasSuccessfulFetch = false;
    let backendWarmupPromise = null;

    const DEFAULT_SUGGESTIONS = [
      "Give me a briefing",
      "Find news on Nvidia",
      "What is trending?"
    ];

    const DEFAULT_TICKER = [
      { topic: "Edition", headline: "A curated front page will appear after the first fetch." },
      { topic: "Coverage", headline: "Lead story, side rail, deeper grid, visuals, and chat stay in sync." },
      { topic: "Assistant", headline: "Ask follow-up questions without leaving the live edition." }
    ];

    const RELIABLE_SOURCES = [
      "Reuters",
      "Associated Press",
      "AP",
      "BBC",
      "Bloomberg",
      "Financial Times",
      "The New York Times",
      "The Wall Street Journal",
      "The Guardian",
      "CNBC"
    ];

    const topicInputs = [
      document.getElementById("topic1"),
      document.getElementById("topic2"),
      document.getElementById("topic3")
    ];

    const heroFetchBtn = document.getElementById("heroFetchBtn");
    const heroInsightsBtn = document.getElementById("heroInsightsBtn");
    const themeToggle = document.getElementById("themeToggle");
    const fetchBtn = document.getElementById("fetchBtn");
    const sampleBtn = document.getElementById("sampleBtn");
    const languageSelect = document.getElementById("languageSelect");
    const statusLine = document.getElementById("statusLine");
    const tickerTrack = document.getElementById("tickerTrack");
    const briefCard = document.getElementById("briefCard");
    const briefText = document.getElementById("briefText");
    const briefTopics = document.getElementById("briefTopics");
    const heroArticle = document.getElementById("heroArticle");
    const secondaryArticles = document.getElementById("secondaryArticles");
    const trendingSection = document.getElementById("trendingSection");
    const trendingSources = document.getElementById("trendingSources");
    const coverageGrid = document.getElementById("coverageGrid");
    const coverageCount = document.getElementById("coverageCount");
    const insightsBand = document.getElementById("insightsBand");
    const resultsContainer = document.getElementById("results-container");
    const agentLogSection = document.getElementById("agentLogSection");
    const agentLogToggle = document.getElementById("agentLogToggle");
    const agentLogContent = document.getElementById("agentLogContent");
    const statArticles = document.getElementById("statArticles");
    const statTopics = document.getElementById("statTopics");
    const statPositive = document.getElementById("statPositive");
    const statLatest = document.getElementById("statLatest");
    const chatFab = document.getElementById("chatFab");
    const chatOverlay = document.getElementById("chatOverlay");
    const chatPanel = document.getElementById("chatPanel");
    const chatCloseBtn = document.getElementById("chatCloseBtn");
    const chatMessages = document.getElementById("chatMessages");
    const suggestionChips = document.getElementById("suggestionChips");
    const quickBriefingBtn = document.getElementById("quickBriefingBtn");
    const chatInput = document.getElementById("chatInput");
    const chatSendBtn = document.getElementById("chatSendBtn");
    const deskSection = document.getElementById("desk");
    const editionStage = document.getElementById("editionStage");
    const heroVisuals = document.getElementById("heroVisuals");
    const heroBlankStack = document.getElementById("heroBlankStack");
    const heroSynthesisProgress = document.getElementById("heroSynthesisProgress");
    const synthesisProgressMessage = document.getElementById("synthesisProgressMessage");
    const cancelRequestBtn = document.getElementById("cancelRequestBtn");
    const wordcloudFrame = document.getElementById("wordcloudFrame");
    const THEME_STORAGE_KEY = "veritas-theme";
    let synthesisMessageTimer = null;
    let synthesisMessageIndex = 0;
    let synthesisMessages = [];
    let activeFetchController = null;
    let activeFetchReset = null;

    function escapeHtml(value) {
      return String(value ?? "")
        .replace(/&/g, "&amp;")
        .replace(/</g, "&lt;")
        .replace(/>/g, "&gt;")
        .replace(/"/g, "&quot;")
        .replace(/'/g, "&#39;");
    }

    function cleanText(value) {
      return String(value ?? "").replace(/\s+/g, " ").trim();
    }

    function delay(ms) {
      return new Promise((resolve) => window.setTimeout(resolve, ms));
    }

    function safeUrl(value) {
      try {
        const url = new URL(String(value || ""));
        if (url.protocol === "http:" || url.protocol === "https:") return url.href;
      } catch (_error) {}
      return "#";
    }

    async function postJSON(url, body, options = {}) {
      const response = await fetch(url, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(body || {}),
        signal: options.signal
      });

      if (!response.ok) {
        let detail = "Request failed";
        try {
          detail = await response.text();
        } catch (_error) {}
        const error = new Error(detail || "Request failed");
        error.status = response.status;
        throw error;
      }

      return response.json();
    }

    async function warmBackend(force = false) {
      if (backendWarmupPromise && !force) return backendWarmupPromise;
      backendWarmupPromise = fetch("/warmup", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: "{}",
        keepalive: true
      })
        .then((response) => response.ok ? response.json().catch(() => ({})) : {})
        .catch(() => ({}));
      return backendWarmupPromise;
    }

    function themeToken(name, fallback) {
      const value = getComputedStyle(document.body).getPropertyValue(name).trim();
      return value || fallback;
    }

    function chartThemeTokens() {
      const isDark = document.body.dataset.theme === "dark";
      return {
        font: themeToken("--text-color", isDark ? "#F5F5F5" : "#1A1A1A"),
        legend: themeToken("--chart-legend", isDark ? "#D8E0DA" : "#1A1A1A"),
        grid: themeToken("--chart-grid", isDark ? "rgba(219, 231, 223, 0.10)" : "rgba(0, 0, 0, 0.10)"),
        zero: themeToken("--chart-zero", isDark ? "rgba(219, 231, 223, 0.18)" : "rgba(0, 0, 0, 0.16)"),
        band: themeToken("--chart-band", isDark ? "rgba(45, 138, 112, 0.18)" : "rgba(45, 138, 112, 0.10)"),
        marker: themeToken("--chart-marker", isDark ? "rgba(102, 179, 151, 0.76)" : "rgba(33, 94, 70, 0.72)"),
        markerLine: themeToken("--chart-marker-line", isDark ? "rgba(15, 21, 17, 0.92)" : "rgba(255,255,255,0.92)"),
        axis: themeToken("--text-color", isDark ? "#F5F5F5" : "#1A1A1A"),
        hoverBg: isDark ? "#1E1E1E" : "#FFFFFF",
        hoverBorder: isDark ? "rgba(255, 255, 255, 0.08)" : "rgba(23, 33, 25, 0.08)"
      };
    }

    function preferredTheme() {
      try {
        const stored = window.localStorage.getItem(THEME_STORAGE_KEY);
        if (stored === "light" || stored === "dark") return stored;
      } catch (_error) {}
      return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light";
    }

    function applyTheme(theme) {
      const nextTheme = theme === "dark" ? "dark" : "light";
      document.body.dataset.theme = nextTheme;
      if (themeToggle) {
        const isDark = nextTheme === "dark";
        themeToggle.textContent = isDark ? "Light Mode" : "Dark Mode";
        themeToggle.setAttribute("aria-pressed", isDark ? "true" : "false");
      }
      try {
        window.localStorage.setItem(THEME_STORAGE_KEY, nextTheme);
      } catch (_error) {}
      window.requestAnimationFrame(syncChartTheme);
    }

    function synthesisMessagesFor(language) {
      return [
        "Initializing Neural Engines...",
        "Translating Global Sources...",
        "Synthesizing Results..."
      ];
    }

    function cancelActiveRequest() {
      if (!activeFetchController) return;
      if (typeof activeFetchReset === "function") activeFetchReset();
      activeFetchController.abort();
    }

    async function fetchJSONWithRetry(url, body, timeoutMs, retry502, retryTimeout, externalSignal) {
      let attemptedRetry = false;
      let attemptedTimeoutRetry = false;
      let activeTimeoutMs = timeoutMs;
      while (true) {
        if (externalSignal && externalSignal.aborted) {
          const abortedError = new Error("aborted");
          abortedError.code = "ABORTED";
          throw abortedError;
        }
        const controller = new AbortController();
        const relayAbort = () => controller.abort();
        if (externalSignal) externalSignal.addEventListener("abort", relayAbort, { once: true });
        const timer = window.setTimeout(() => controller.abort(), activeTimeoutMs);
        try {
          const response = await fetch(url, {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify(body || {}),
            signal: controller.signal
          });

          if (response.status === 502 && retry502 && !attemptedRetry) {
            attemptedRetry = true;
            setStatus("Tunnel warming up. Retrying once automatically...", false);
            await warmBackend(true);
            await delay(900);
            continue;
          }

          if (!response.ok) {
            const detail = await response.text().catch(() => "Request failed");
            const error = new Error(detail || "Request failed");
            error.status = response.status;
            throw error;
          }

          return response.json();
        } catch (error) {
          if (externalSignal && externalSignal.aborted) {
            const abortedError = new Error("aborted");
            abortedError.code = "ABORTED";
            throw abortedError;
          }
          if (error.name === "AbortError") {
            if (retryTimeout && !attemptedTimeoutRetry) {
              attemptedTimeoutRetry = true;
              activeTimeoutMs = Math.max(activeTimeoutMs, 45000);
              setStatus("First load is warming the research engine. Holding the connection a little longer...", false);
              await warmBackend(true);
              await delay(1200);
              continue;
            }
            const timeoutError = new Error("timeout");
            timeoutError.code = "TIMEOUT";
            throw timeoutError;
          }
          if (error.status === 502 && retry502 && !attemptedRetry) {
            attemptedRetry = true;
            setStatus("Tunnel warming up. Retrying once automatically...", false);
            await warmBackend(true);
            await delay(900);
            continue;
          }
          throw error;
        } finally {
          window.clearTimeout(timer);
          if (externalSignal) externalSignal.removeEventListener("abort", relayAbort);
        }
      }
    }

    function sourceFor(article) {
      return cleanText(article && article.source) || "Unknown Source";
    }

    function topicFor(article) {
      return cleanText(article && article.topic) || "General";
    }

    function publishedFor(article) {
      return cleanText(article && article.published_display) || "Latest";
    }

    function articleDateFor(article) {
      const candidates = [
        article && article.published_at,
        article && article.published,
        article && article.pubDate,
        article && article.published_display
      ];

      for (const candidate of candidates) {
        const value = cleanText(candidate);
        if (!value) continue;
        const date = new Date(value);
        if (!Number.isNaN(date.getTime())) return date;
      }

      return null;
    }

    function relativeTimeFromDate(date, fallback) {
      if (!(date instanceof Date) || Number.isNaN(date.getTime())) return cleanText(fallback) || "Live";
      const diff = Date.now() - date.getTime();
      if (!Number.isFinite(diff) || diff < 0) return cleanText(fallback) || "Live";

      const minute = 60 * 1000;
      const hour = 60 * minute;
      const day = 24 * hour;

      if (diff < minute) return "Just now";
      if (diff < hour) return `${Math.max(1, Math.floor(diff / minute))}m ago`;
      if (diff < day) return `${Math.max(1, Math.floor(diff / hour))}h ago`;
      return `${Math.max(1, Math.floor(diff / day))}d ago`;
    }

    function relativeTimeFor(article) {
      const date = articleDateFor(article);
      return relativeTimeFromDate(date, publishedFor(article));
    }

    function headlineFor(article) {
      return cleanText(article && (article.translated_headline || article.headline));
    }

    function summaryFor(article, limit) {
      const text = cleanText(article && (article.summary || article.headline || article.translated_headline));
      if (!text) return "";
      if (text.length <= limit) return text;
      return `${text.slice(0, Math.max(limit - 1, 0)).trimEnd()}...`;
    }

    function filteredArticles(articles) {
      return (articles || []).filter((article) => safeUrl(article && article.link) !== "#" && headlineFor(article));
    }

    function sentimentKey(sentiment) {
      const value = cleanText(sentiment).toLowerCase();
      if (value === "positive") return "positive";
      if (value === "negative") return "negative";
      return "neutral";
    }

    function sentimentLabel(sentiment) {
      const key = sentimentKey(sentiment);
      return key.charAt(0).toUpperCase() + key.slice(1);
    }

    function sentimentBadgeMarkup(sentiment) {
      const key = sentimentKey(sentiment);
      return `<span class="sentiment ${key}">${escapeHtml(sentimentLabel(sentiment))}</span>`;
    }

    function readingTimeFor(article) {
      const content = `${headlineFor(article)} ${summaryFor(article, 280)}`.trim();
      const words = content.split(/\s+/).filter(Boolean).length || 1;
      return `Reading Time: ${Math.max(1, Math.ceil(words / 220))} min`;
    }

    function sourceReliabilityFor(source) {
      const normalized = cleanText(source).toLowerCase();
      const match = RELIABLE_SOURCES.some((item) => normalized.includes(item.toLowerCase()));
      return match ? "Source Reliability: High" : "Source Reliability: Tracked";
    }

    function setStatus(message, isError) {
      statusLine.textContent = cleanText(message);
      statusLine.classList.toggle("is-error", Boolean(isError));
    }

    function setFetchLoading(isLoading) {
      fetchBtn.classList.toggle("is-loading", Boolean(isLoading));
      heroFetchBtn.classList.toggle("is-loading", Boolean(isLoading));
      sampleBtn.classList.toggle("is-loading", Boolean(isLoading));
      fetchBtn.textContent = isLoading ? "Fetching..." : "Fetch News";
      heroFetchBtn.textContent = isLoading ? "Building..." : "Build Live Edition";
      fetchBtn.disabled = Boolean(isLoading);
      heroFetchBtn.disabled = Boolean(isLoading);
      sampleBtn.disabled = Boolean(isLoading);
      languageSelect.disabled = Boolean(isLoading);
      topicInputs.forEach((input) => { input.disabled = Boolean(isLoading); });
    }

    function setSynthesisProgress(isVisible, language) {
      if (!heroBlankStack || !heroSynthesisProgress || !synthesisProgressMessage) return;
      window.clearInterval(synthesisMessageTimer);
      synthesisMessageTimer = null;

      if (isVisible) {
        synthesisMessages = synthesisMessagesFor(language);
        synthesisMessageIndex = 0;
        synthesisProgressMessage.textContent = synthesisMessages[synthesisMessageIndex];
        heroBlankStack.hidden = true;
        heroSynthesisProgress.hidden = false;
        if (heroVisuals) heroVisuals.classList.add("is-synthesizing");
        synthesisMessageTimer = window.setInterval(() => {
          synthesisMessageIndex = (synthesisMessageIndex + 1) % synthesisMessages.length;
          synthesisProgressMessage.textContent = synthesisMessages[synthesisMessageIndex];
        }, 12000);
        return;
      }

      heroSynthesisProgress.hidden = true;
      heroBlankStack.hidden = false;
      if (heroVisuals) heroVisuals.classList.remove("is-synthesizing");
      synthesisMessageIndex = 0;
      synthesisMessages = synthesisMessagesFor(languageSelect ? languageSelect.value : "English");
      synthesisProgressMessage.textContent = synthesisMessages[0];
    }

    function setChatLoading(isLoading) {
      chatSendBtn.classList.toggle("is-loading", Boolean(isLoading));
      chatSendBtn.textContent = isLoading ? "Sending..." : "Ask";
      chatSendBtn.disabled = Boolean(isLoading);
      quickBriefingBtn.disabled = Boolean(isLoading);
      chatInput.disabled = Boolean(isLoading);
    }

    function metaDotMarkup() {
      return '<span class="meta-dot" aria-hidden="true"></span>';
    }

    function updateCoverageCount(count) {
      if (count > 0) {
        coverageCount.textContent = `${count} stories`;
        coverageCount.classList.add("is-visible");
      } else {
        coverageCount.textContent = "";
        coverageCount.classList.remove("is-visible");
      }
    }

    function quietFeatureMarkup() {
      return `
        <div class="quiet-card">
          <div class="quiet-panel">
            <div>
              <span class="quiet-kicker">Edition Preview</span>
              <h3 class="quiet-title">The first fetch turns this into a composed lead story.</h3>
              <p class="quiet-copy">Use the desk above to build a calmer front page with a lead article, a side rail, deeper coverage, visuals, and a reasoning trace.</p>
            </div>
          </div>
        </div>
      `;
    }

    function previewHeroCardsMarkup() {
      return `
        <article class="blank-card blank-card--one">
          <div class="blank-card-top">
            <span class="blank-pill short"></span>
            <div class="blank-chip-row">
              <span class="blank-chip"></span>
              <span class="blank-chip"></span>
            </div>
          </div>
          <div class="blank-line xl"></div>
          <div class="blank-line lg"></div>
          <div class="blank-line md"></div>
          <div class="blank-card-frame">
            <div class="blank-card-box"></div>
            <div class="blank-card-box"></div>
          </div>
        </article>
        <article class="blank-card blank-card--two">
          <div class="blank-card-top">
            <span class="blank-pill mid"></span>
            <div class="blank-chip-row">
              <span class="blank-chip"></span>
              <span class="blank-chip"></span>
              <span class="blank-chip"></span>
            </div>
          </div>
          <div class="blank-line xl"></div>
          <div class="blank-line lg"></div>
          <div class="blank-line sm"></div>
          <div class="blank-card-footer">
            <span class="blank-card-dot"></span>
            <div class="blank-line md"></div>
          </div>
        </article>
        <article class="blank-card blank-card--three">
          <div class="blank-card-top">
            <span class="blank-pill long"></span>
            <div class="blank-chip-row">
              <span class="blank-chip"></span>
              <span class="blank-chip"></span>
            </div>
          </div>
          <div class="blank-line xl"></div>
          <div class="blank-line lg"></div>
          <div class="blank-line md"></div>
          <div class="blank-line sm"></div>
          <div class="blank-card-frame">
            <div class="blank-card-box"></div>
            <div class="blank-card-box"></div>
          </div>
          <div class="blank-card-footer">
            <span class="blank-card-dot"></span>
            <div class="blank-line lg"></div>
          </div>
        </article>
      `;
    }

    function editionTimestamp(date = new Date()) {
      try {
        return new Intl.DateTimeFormat(undefined, {
          month: "short",
          day: "numeric",
          hour: "numeric",
          minute: "2-digit"
        }).format(date);
      } catch (_error) {
        return cleanText(date && date.toString()) || "Now";
      }
    }

    function extractHeroCategories(articles, topics) {
      const values = [];
      (articles || []).forEach((article) => {
        const direct = [
          cleanText(article && article.category),
          cleanText(article && article.section),
          cleanText(article && article.topic)
        ].filter(Boolean);
        const multi = Array.isArray(article && article.categories)
          ? article.categories.map((value) => cleanText(value)).filter(Boolean)
          : [];
        [...direct, ...multi].forEach((value) => {
          if (value && !values.includes(value)) values.push(value);
        });
      });

      (topics || []).map((value) => cleanText(value)).filter(Boolean).forEach((value) => {
        if (!values.includes(value)) values.push(value);
      });

      return values.slice(0, 3);
    }

    function liveHeroCardsMarkup(articles, topics, createdAt) {
      const items = filteredArticles(articles);
      const lead = items[0] || {};
      const categories = extractHeroCategories(items, topics);
      const categoryList = categories.length ? categories : ["General"];
      const articleCount = items.length;
      const stamp = editionTimestamp(createdAt);

      return `
        <article class="blank-card blank-card--one is-live">
          <div class="hero-card-shell">
            <div class="hero-card-meta">
              <span class="hero-card-kicker">Edition Ledger</span>
              <span class="hero-card-stamp">${escapeHtml(sourceFor(lead) || "Live Desk")}</span>
            </div>
            <div class="hero-card-stat-grid">
              <div class="hero-card-stat">
                <p class="hero-card-stat-label">Stories</p>
                <p class="hero-card-stat-value">${escapeHtml(String(articleCount || 0))}</p>
                <p class="hero-card-stat-copy">Stories accepted into the current edition.</p>
              </div>
              <div class="hero-card-stat">
                <p class="hero-card-stat-label">Edition Time</p>
                <p class="hero-card-stat-value">${escapeHtml(stamp)}</p>
                <p class="hero-card-stat-copy">Timestamp when this edition card deck resolved.</p>
              </div>
            </div>
            <p class="hero-card-note">The full coverage grid below expands this count into a source-by-source reading path.</p>
          </div>
        </article>
        <article class="blank-card blank-card--two is-live">
          <div class="hero-card-shell">
            <div class="hero-card-meta">
              <span class="hero-card-kicker">Edition Signals</span>
              <span class="hero-card-stamp">${escapeHtml(stamp)}</span>
            </div>
            <div class="hero-card-list">
              <p class="hero-card-list-label">Categories</p>
              <div class="hero-card-badge-row">
                ${categoryList.map((category) => `<span class="hero-card-badge">${escapeHtml(category)}</span>`).join("")}
              </div>
            </div>
            <p class="hero-card-note">Three live angles pulled from the fetched JSON are pinned here to frame the edition before you move into the full desk.</p>
          </div>
        </article>
        <article class="blank-card blank-card--three is-live">
          <div class="hero-card-shell">
            <div class="hero-card-meta">
              <span class="hero-card-kicker">Front Page</span>
              <span class="hero-card-stamp">${escapeHtml(relativeTimeFor(lead))}</span>
            </div>
            <div>
              <h3 class="hero-card-title">${escapeHtml(headlineFor(lead) || "Live edition ready")}</h3>
              <p class="hero-card-summary">${escapeHtml(summaryFor(lead, 250) || "The current lead story is now surfaced and ready to read in the main edition below.")}</p>
            </div>
            <div class="hero-card-footer">
              <p class="hero-card-caption">The lead story below anchors the full edition, with the rail and coverage grid continuing the briefing.</p>
              <button class="hero-card-arrow" id="heroEditionArrow" type="button" aria-label="Scroll to more news">
                <svg viewBox="0 0 24 24" aria-hidden="true">
                  <path d="M12 5v14"></path>
                  <path d="m6 13 6 6 6-6"></path>
                </svg>
              </button>
            </div>
          </div>
        </article>
      `;
    }

    function renderHeroPreviewCards() {
      if (!heroBlankStack) return;
      heroBlankStack.classList.remove("is-loaded");
      heroBlankStack.innerHTML = previewHeroCardsMarkup();
    }

    function renderHeroEditionCards(articles, topics, options = {}) {
      if (!heroBlankStack) return;
      const items = filteredArticles(articles);
      if (!items.length) {
        renderHeroPreviewCards();
        return;
      }

      const animate = options.animate !== false;
      heroBlankStack.classList.remove("is-loaded");
      heroBlankStack.innerHTML = liveHeroCardsMarkup(items, topics || currentTopics, options.createdAt || new Date());
      if (!animate) {
        heroBlankStack.classList.add("is-loaded");
        return;
      }
      window.requestAnimationFrame(() => {
        window.requestAnimationFrame(() => heroBlankStack.classList.add("is-loaded"));
      });
    }

    function quietRailMarkup() {
      const rows = [
        { title: "Secondary stories appear here after the first fetch." },
        { title: "Each item keeps only source, time, headline, and summary." },
        { title: "More stories continue below in the coverage grid." }
      ];

      return `
        <div class="quiet-rail">
          ${rows.map((row) => `
            <div class="quiet-rail-row">
              <div class="quiet-title-small">${escapeHtml(row.title)}</div>
            </div>
          `).join("")}
        </div>
      `;
    }

    function quietCoverageMarkup() {
      const slots = [
        "Global Perspective: Cross-references international wires to identify localized impacts of global trends.",
        "Temporal Analysis: Tracks how storylines have evolved over the last 24 hours to separate breaking news from noise.",
        "Source Integrity: Weights reporting based on historical accuracy and publisher bias metrics."
      ];

      return slots.map((copy) => `
        <article class="surface interactive coverage-card">
          <p class="coverage-summary">${escapeHtml(copy)}</p>
        </article>
      `).join("");
    }

    function connectionRetryMarkup() {
      return `
        <div class="quiet-card">
          <div class="quiet-panel">
            <div>
              <span class="quiet-kicker">Connection Refined</span>
              <h3 class="quiet-title">Connection Refined - Please Retry</h3>
              <p class="quiet-copy">The desk could not complete this pass. Please retry once the connection settles.</p>
            </div>
          </div>
        </div>
      `;
    }

    function deferredFeatureMarkup() {
      return `
        <div class="quiet-card">
          <div class="quiet-panel">
            <div>
              <span class="quiet-kicker">Warming Up</span>
              <h3 class="quiet-title">The desk is still assembling the current edition.</h3>
              <p class="quiet-copy">This topic mix is taking longer than usual to resolve. Retry in a few seconds or narrow the topic set for a faster first pass.</p>
            </div>
          </div>
        </div>
      `;
    }

    function renderDeferredState() {
      heroArticle.innerHTML = deferredFeatureMarkup();
      secondaryArticles.innerHTML = quietRailMarkup();
      trendingSection.classList.remove("is-visible");
      trendingSources.innerHTML = "";
      coverageGrid.innerHTML = quietCoverageMarkup();
      updateCoverageCount(0);
    }

    function renderConnectionRetryState() {
      heroArticle.innerHTML = connectionRetryMarkup();
      secondaryArticles.innerHTML = quietRailMarkup();
      trendingSection.classList.remove("is-visible");
      trendingSources.innerHTML = "";
      coverageGrid.innerHTML = quietCoverageMarkup();
      updateCoverageCount(0);
    }

    function loadingFeatureMarkup() {
      return `
        <div class="loading-lead">
          <div class="loading-lead-copy">
            <div class="skeleton-meta">
              <div class="skeleton-pill wide"></div>
              <div class="skeleton-pill mid"></div>
              <div class="skeleton-pill wide"></div>
            </div>
            <div class="skeleton-line xl"></div>
            <div class="skeleton-line lg"></div>
            <div class="skeleton-meta">
              <div class="skeleton-pill wide"></div>
              <div class="skeleton-pill wide"></div>
              <div class="skeleton-pill mid"></div>
            </div>
            <div class="skeleton-line md"></div>
            <div class="skeleton-line sm"></div>
          </div>
        </div>
      `;
    }

    function loadingRailMarkup() {
      return `
        <div class="loading-rail">
          ${Array.from({ length: 4 }).map((_, index) => `
            <div class="loading-rail-row">
              <div class="skeleton-meta">
                <div class="skeleton-pill wide"></div>
                <div class="skeleton-pill mid"></div>
                <div class="skeleton-pill mid"></div>
              </div>
              <div class="skeleton-line ${index % 2 === 0 ? "lg" : "md"}"></div>
              <div class="skeleton-line sm"></div>
            </div>
          `).join("")}
        </div>
      `;
    }

    function loadingCoverageMarkup() {
      return Array.from({ length: 3 }).map(() => `
        <article class="surface loading-coverage">
          <div class="skeleton-meta">
            <div class="skeleton-pill wide"></div>
            <div class="skeleton-pill mid"></div>
          </div>
          <div class="skeleton-line xl"></div>
          <div class="skeleton-line lg"></div>
          <div class="skeleton-line md"></div>
          <div class="skeleton-line sm"></div>
        </article>
      `).join("");
    }

    function renderQuietState() {
      heroArticle.innerHTML = quietFeatureMarkup();
      secondaryArticles.innerHTML = quietRailMarkup();
      trendingSection.classList.remove("is-visible");
      trendingSources.innerHTML = "";
      coverageGrid.innerHTML = quietCoverageMarkup();
      updateCoverageCount(0);
    }

    function renderLoadingState() {
      heroArticle.innerHTML = loadingFeatureMarkup();
      secondaryArticles.innerHTML = loadingRailMarkup();
      trendingSection.classList.remove("is-visible");
      trendingSources.innerHTML = "";
      coverageGrid.innerHTML = loadingCoverageMarkup();
      updateCoverageCount(0);
    }

    function renderLeadArticle(article) {
      heroArticle.innerHTML = `
        <div class="story-card news-card">
          <div class="lead-story-layout">
            <div class="lead-story-copy">
              <div class="lead-kicker">
                <div class="lead-kicker-left">
                  <span class="meta-pill">${escapeHtml(sourceFor(article))}</span>
                  <span class="meta-pill">${escapeHtml(topicFor(article))}</span>
                </div>
              </div>
              <div class="lead-body">
                <p class="lead-overline">${escapeHtml(relativeTimeFor(article))} ${metaDotMarkup()} ${escapeHtml(publishedFor(article))}</p>
                <h2 class="lead-headline">
                  <a href="${escapeHtml(safeUrl(article.link))}" target="_blank" rel="noopener noreferrer">${escapeHtml(headlineFor(article))}</a>
                </h2>
                <div class="lead-facts">
                  <span class="lead-fact">${escapeHtml(readingTimeFor(article))}</span>
                  <span class="lead-fact">${escapeHtml(sourceReliabilityFor(sourceFor(article)))}</span>
                  ${sentimentBadgeMarkup(article.sentiment)}
                </div>
                <p class="lead-summary">${escapeHtml(summaryFor(article, 360))}</p>
              </div>
              <div class="lead-note">
                This lead story is the anchor for the current edition. Open the full article for source context, then use the assistant for follow-up synthesis.
              </div>
            </div>
          </div>
        </div>
      `;
    }

    function renderRailArticles(articles) {
      const items = filteredArticles(articles).slice(0, 5);
      if (!items.length) {
        secondaryArticles.innerHTML = quietRailMarkup();
        return;
      }

      secondaryArticles.innerHTML = `
        ${items.map((article, index) => `
        <article class="rail-item news-card" style="animation-delay:${index * 70}ms;">
          <div class="rail-body">
            <div class="rail-topline">
              <div class="rail-meta">
                <span class="rail-source">${escapeHtml(sourceFor(article))}</span>
                ${metaDotMarkup()}
                <span>${escapeHtml(relativeTimeFor(article))}</span>
              </div>
            </div>
            <div class="rail-meta">
              <span>${escapeHtml(topicFor(article))}</span>
              ${sentimentBadgeMarkup(article.sentiment)}
            </div>
            <h3 class="rail-headline">
              <a href="${escapeHtml(safeUrl(article.link))}" target="_blank" rel="noopener noreferrer">${escapeHtml(headlineFor(article))}</a>
            </h3>
            <div class="rail-summary">${escapeHtml(summaryFor(article, 120))}</div>
          </div>
        </article>
      `).join("")}
      `;
    }

    function renderTrendingSources(articles) {
      const unique = [];
      filteredArticles(articles).forEach((article) => {
        const source = sourceFor(article);
        if (source && !unique.includes(source)) unique.push(source);
      });

      if (!unique.length) {
        trendingSection.classList.remove("is-visible");
        trendingSources.innerHTML = "";
        return;
      }

      trendingSection.classList.add("is-visible");
      trendingSources.innerHTML = unique.slice(0, 12).map((source) => `
        <span class="trending-pill">${escapeHtml(source)}</span>
      `).join("");
    }

    function renderCoverage(articles) {
      const items = filteredArticles(articles).slice(6);
      if (!items.length) {
        coverageGrid.innerHTML = quietCoverageMarkup();
        updateCoverageCount(0);
        return;
      }

      updateCoverageCount(items.length);
      coverageGrid.innerHTML = items.map((article, index) => `
        <article class="surface interactive coverage-card news-card" style="animation-delay:${index * 60}ms;">
          <div class="coverage-meta">
            <span class="coverage-source">${escapeHtml(sourceFor(article))}</span>
            ${sentimentBadgeMarkup(article.sentiment)}
            ${metaDotMarkup()}
            <span>${escapeHtml(relativeTimeFor(article))}</span>
          </div>
          <h3 class="coverage-headline">
            <a href="${escapeHtml(safeUrl(article.link))}" target="_blank" rel="noopener noreferrer">${escapeHtml(headlineFor(article))}</a>
          </h3>
          <p class="coverage-summary">${escapeHtml(summaryFor(article, 160))}</p>
          <a class="coverage-link" href="${escapeHtml(safeUrl(article.link))}" target="_blank" rel="noopener noreferrer">Read more</a>
        </article>
      `).join("");
    }

    function renderArticles(articles) {
      const items = filteredArticles(articles);
      if (!items.length) {
        renderQuietState();
        return;
      }

      renderLeadArticle(items[0]);
      renderRailArticles(items.slice(1, 6));
      renderTrendingSources(items);
      renderCoverage(items);
    }

    function renderTicker(articles) {
      const items = filteredArticles(articles).slice(0, 10).map((article) => ({
        topic: topicFor(article),
        headline: headlineFor(article)
      }));
      const payload = items.length ? items : DEFAULT_TICKER;
      const markup = payload.map((item) => `
        <span class="ticker-item">
          <strong>${escapeHtml(item.topic)}</strong>
          <span>${escapeHtml(item.headline)}</span>
        </span>
      `).join("");
      tickerTrack.innerHTML = markup + markup;
    }

    function renderBriefing(briefing, topics) {
      const cleanBrief = cleanText(briefing);
      if (!cleanBrief) {
        briefCard.hidden = true;
        briefCard.classList.remove("is-visible");
        briefText.textContent = "";
        briefTopics.innerHTML = "";
        return;
      }

      briefCard.hidden = false;
      briefCard.classList.add("is-visible");
      briefText.textContent = cleanBrief;
      briefTopics.innerHTML = (topics || []).filter(Boolean).map((topic) => `
        <span class="brief-topic">${escapeHtml(cleanText(topic))}</span>
      `).join("");
    }

    function clearVisualFrames() {
      if (wordcloudFrame) {
        wordcloudFrame.innerHTML = '<div class="wordcloud-stage is-loading"></div>';
      }
      ["pie-chart", "bar-chart", "scatter-chart"].forEach((id) => {
        const el = document.getElementById(id);
        if (!el) return;
        if (typeof Plotly !== "undefined") {
          try { Plotly.purge(el); } catch (_error) {}
        }
        el.innerHTML = '<div class="visual-empty"></div>';
      });
    }

    function toDateValue(value) {
      const date = value instanceof Date ? value : new Date(value);
      return Number.isNaN(date.getTime()) ? null : date;
    }

    function gradientPalette(length) {
      const start = [33, 94, 70];
      const end = [98, 111, 125];
      const total = Math.max(length - 1, 1);
      return Array.from({ length: Math.max(length, 1) }).map((_, index) => {
        const ratio = index / total;
        const rgb = start.map((channel, i) => Math.round(channel + ((end[i] - channel) * ratio)));
        return `rgb(${rgb[0]}, ${rgb[1]}, ${rgb[2]})`;
      });
    }

    function relativeTickConfig(values, tickColor) {
      const parsed = (values || []).map((value) => ({ raw: value, date: toDateValue(value) })).filter((item) => item.date);
      return {
        showticklabels: false,
        tickfont: { color: tickColor || "#1A1A1A", size: 11 },
        ticklabeloverflow: "hide past domain",
        automargin: true
      };
    }

    function scatterThemeShapes(_theme) {
      return [];
    }

    function scrubPlotlySurface(el) {
      if (!el) return;
      el.style.background = "transparent";
      const selectors = [
        ".bg",
        ".plotbg",
        ".bglayer rect",
        ".cartesianlayer .subplot rect.bg",
        ".main-svg rect.bg",
        ".main-svg",
        ".svg-container",
        ".plot-container"
      ];
      el.querySelectorAll(selectors.join(",")).forEach((node) => {
        if (node instanceof SVGElement) {
          node.setAttribute("fill", "transparent");
          node.setAttribute("stroke", "transparent");
        }
        if (node.style) {
          node.style.background = "transparent";
          node.style.fill = "transparent";
          node.style.stroke = "transparent";
          node.style.boxShadow = "none";
        }
      });
    }

    function syncChartTheme() {
      if (typeof Plotly === "undefined") return;
      const theme = chartThemeTokens();
      const commonLayout = {
        paper_bgcolor: "rgba(0,0,0,0)",
        plot_bgcolor: "rgba(0,0,0,0)",
        font: { family: "Inter, sans-serif", size: 11, color: theme.font },
        legend: {
          bgcolor: "rgba(0,0,0,0)",
          font: { family: "Inter, sans-serif", size: 12, color: theme.legend }
        }
      };

      const pieChart = document.getElementById("pie-chart");
      if (pieChart && Array.isArray(pieChart.data) && pieChart.data.length) {
        Plotly.restyle(pieChart, {
          textinfo: "percent+label",
          textposition: "inside",
          insidetextfont: [{ color: "#FFFFFF", family: "Inter, sans-serif", size: 16 }],
          textfont: [{ color: "#FFFFFF", family: "Inter, sans-serif", size: 16 }],
          hoverlabel: [{
            bgcolor: theme.hoverBg,
            bordercolor: theme.hoverBorder,
            font: { color: theme.font, family: "Inter, sans-serif" }
          }]
        }).catch(() => {});
        Plotly.relayout(pieChart, Object.assign({}, commonLayout, {
          automargin: false,
          margin: { t: 0, b: 0, l: 0, r: 0 },
          showlegend: true,
          legend: {
            x: 1,
            y: 0.5,
            xanchor: "left",
            yanchor: "middle",
            orientation: "v",
            bgcolor: "rgba(0,0,0,0)",
            font: { family: "Inter, sans-serif", size: 12, color: theme.legend }
          },
          uniformtext: { minsize: 12, mode: "hide" }
        })).then(() => scrubPlotlySurface(pieChart)).catch(() => {});
      }

      const barChart = document.getElementById("bar-chart");
      if (barChart && Array.isArray(barChart.data) && barChart.data.length) {
        Plotly.relayout(barChart, Object.assign({}, commonLayout, {
          margin: { t: 0, b: 0, l: 0, r: 0 },
          xaxis: Object.assign({}, barChart.layout && barChart.layout.xaxis ? barChart.layout.xaxis : {}, {
            showgrid: true,
            gridcolor: theme.grid,
            zeroline: false,
            color: theme.font,
            tickfont: { color: theme.font, size: 11 },
            title: ""
          }),
          yaxis: Object.assign({}, barChart.layout && barChart.layout.yaxis ? barChart.layout.yaxis : {}, {
            showgrid: false,
            color: theme.font,
            tickfont: { color: theme.font, size: 11 },
            title: ""
          })
        })).then(() => scrubPlotlySurface(barChart)).catch(() => {});
      }

      const scatterChart = document.getElementById("scatter-chart");
      if (scatterChart && Array.isArray(scatterChart.data) && scatterChart.data.length) {
        const scatterXValues = Array.isArray(scatterChart.data[0] && scatterChart.data[0].x) ? scatterChart.data[0].x : [];
        Plotly.restyle(scatterChart, {
          "marker.color": theme.marker,
          "marker.line.color": theme.markerLine
        }).catch(() => {});
        Plotly.relayout(scatterChart, Object.assign({}, commonLayout, {
          margin: { t: 0, b: 0, l: 0, r: 0 },
          xaxis: Object.assign(
            {},
            scatterChart.layout && scatterChart.layout.xaxis ? scatterChart.layout.xaxis : {},
            relativeTickConfig(scatterXValues, theme.font),
            {
              title: "",
              showgrid: true,
              gridcolor: theme.grid,
              zeroline: false,
              color: theme.font,
              tickfont: { color: theme.font, size: 11 }
            }
          ),
          yaxis: Object.assign({}, scatterChart.layout && scatterChart.layout.yaxis ? scatterChart.layout.yaxis : {}, {
            title: "",
            showgrid: true,
            gridcolor: theme.grid,
            zeroline: false,
            color: theme.font,
            tickfont: { color: theme.font, size: 11 }
          }),
          shapes: scatterThemeShapes(theme)
        })).then(() => scrubPlotlySurface(scatterChart)).catch(() => {});
      }
    }

    function renderWordcloud(dataUrl) {
      if (!wordcloudFrame) return;
      if (!cleanText(dataUrl)) {
        wordcloudFrame.innerHTML = '<div class="wordcloud-stage is-loading"></div>';
        return;
      }

      wordcloudFrame.innerHTML = `<div class="wordcloud-stage"><img class="wordcloud-image" src="${escapeHtml(dataUrl)}" alt="Word cloud visualization" /></div>`;
    }

    async function loadVisuals(retry = true) {
      if (!originalArticles.length) return;
      if (typeof Plotly === "undefined") {
        if (retry) setTimeout(() => loadVisuals(false), 500);
        return;
      }
      try {
        const theme = chartThemeTokens();
        const res = await fetch("/visuals", {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ articles: originalArticles })
        });
        const data = await res.json();

        renderWordcloud(data.wordcloud || "");

        const chartConfigs = [
          { key: "pie", id: "pie-chart" },
          { key: "bar", id: "bar-chart" },
          { key: "scatter", id: "scatter-chart" }
        ];

        for (const cfg of chartConfigs) {
          const figure = data[cfg.key];
          const el = document.getElementById(cfg.id);
          if (!figure || !el) continue;

          let fig = typeof figure === "string" ? JSON.parse(figure) : figure;
          const greenPalette = ["#1D5640", "#2A7255", "#418A6D", "#5BA286", "#86B8A2"];

          if (cfg.key === "pie" && Array.isArray(fig.data)) {
            const pieColorMap = {
              positive: "#356746",
              negative: "#B25755",
              neutral: "#708090"
            };
            fig = Object.assign({}, fig, {
              data: fig.data.map((trace) => Object.assign({}, trace, {
                marker: Object.assign({}, trace.marker || {}, {
                  colors: Array.isArray(trace.labels)
                    ? trace.labels.map((label) => {
                        const key = cleanText(label).toLowerCase();
                        return pieColorMap[key] || "#708090";
                      })
                    : ["#356746", "#B25755", "#708090"],
                  line: { color: "rgba(255,255,255,0.9)", width: 1.2 }
                }),
                sort: false,
                automargin: true,
                textinfo: "percent+label",
                textposition: "inside",
                insidetextorientation: "auto",
                insidetextfont: { color: "#FFFFFF", family: "Inter, sans-serif", size: 16 },
                textfont: { color: "#FFFFFF", family: "Inter, sans-serif", size: 16 },
                hoverlabel: { bgcolor: theme.hoverBg, bordercolor: theme.hoverBorder, font: { color: theme.font, family: "Inter, sans-serif" } }
              }))
            });
          }

          if (cfg.key === "bar" && Array.isArray(fig.data)) {
            const shades = gradientPalette(Array.isArray(fig.data[0] && fig.data[0].y) ? fig.data[0].y.length : 6);
            fig = Object.assign({}, fig, {
              data: fig.data.map((trace) => Object.assign({}, trace, {
                marker: Object.assign({}, trace.marker || {}, {
                  color: Array.isArray(trace.y)
                    ? trace.y.map((_, index) => shades[index % shades.length])
                    : shades[2],
                  line: { color: "rgba(255,255,255,0.9)", width: 1.2 }
                })
              }))
            });
          }

          if (cfg.key === "scatter" && Array.isArray(fig.data)) {
            fig = Object.assign({}, fig, {
              data: fig.data.map((trace) => Object.assign({}, trace, {
                customdata: Array.isArray(trace.x)
                  ? trace.x.map((value) => {
                      const date = toDateValue(value);
                      return relativeTimeFromDate(date, "Live");
                    })
                  : [],
                marker: Object.assign({}, trace.marker || {}, {
                  size: 11,
                  color: theme.marker,
                  line: { width: 1.2, color: theme.markerLine }
                }),
                hovertemplate: "%{text}<br>%{customdata}<br>Score: %{y:.2f}<extra></extra>"
              }))
            });
          }

          const layout = Object.assign({}, fig.layout || {}, {
            paper_bgcolor: "rgba(0,0,0,0)",
            plot_bgcolor: "rgba(0,0,0,0)",
            automargin: false,
            margin: { t: 0, b: 0, l: 0, r: 0 },
            font: { family: "Inter, sans-serif", size: 11, color: theme.font },
            legend: {
              bgcolor: "rgba(0,0,0,0)",
              font: { family: "Inter, sans-serif", size: 12, color: theme.legend }
            },
            title: { text: "" },
            autosize: true
          });

          if (cfg.key === "pie") {
            layout.automargin = false;
            layout.margin = { t: 0, b: 0, l: 0, r: 0 };
            layout.showlegend = true;
            layout.legend = Object.assign({}, layout.legend || {}, {
              x: 1,
              y: 0.5,
              xanchor: "left",
              yanchor: "middle",
              orientation: "v",
              bgcolor: "rgba(0,0,0,0)",
              font: { family: "Inter, sans-serif", size: 12, color: theme.legend }
            });
            layout.uniformtext = { minsize: 12, mode: "hide" };
            layout.annotations = [];
          }

          if (cfg.key === "bar") {
            layout.xaxis = Object.assign({}, layout.xaxis || {}, {
              showgrid: true,
              gridcolor: theme.grid,
              zeroline: false,
              color: theme.font,
              tickfont: { color: theme.font, size: 11 },
              title: ""
            });
            layout.yaxis = Object.assign({}, layout.yaxis || {}, {
              showgrid: false,
              color: theme.font,
              tickfont: { color: theme.font, size: 11 },
              title: ""
            });
          }

          if (cfg.key === "scatter") {
            const xValues = Array.isArray(fig.data && fig.data[0] && fig.data[0].x) ? fig.data[0].x : [];
            layout.xaxis = Object.assign({}, layout.xaxis || {}, relativeTickConfig(xValues, theme.font), {
              title: "",
              showgrid: true,
              gridcolor: theme.grid,
              zeroline: false,
              color: theme.font,
              tickfont: { color: theme.font, size: 11 }
            });
            layout.yaxis = Object.assign({}, layout.yaxis || {}, {
              title: "",
              showgrid: true,
              gridcolor: theme.grid,
              zeroline: false,
              color: theme.font,
              tickfont: { color: theme.font, size: 11 }
            });
            layout.shapes = scatterThemeShapes(theme);
          }

          Plotly.react(el, fig.data || [], layout, {
            displayModeBar: false,
            responsive: true
          }).then(() => scrubPlotlySurface(el));
        }
      } catch (error) {
        console.error("Visuals loading failed:", error);
      }
    }

    function renderAgentLog(data) {
      const topics = Object.keys(data || {});
      if (!topics.length) {
        agentLogSection.hidden = true;
        agentLogSection.classList.remove("is-visible", "is-open");
        agentLogContent.innerHTML = "";
        agentLogOpen = false;
        agentLogToggle.setAttribute("aria-expanded", "false");
        return;
      }

      agentLogSection.hidden = false;
      agentLogSection.classList.add("is-visible");
      agentLogContent.innerHTML = topics.map((topic) => {
        const entries = Array.isArray(data[topic]) ? data[topic] : [];
        return `
          <article class="surface interactive log-card">
            <h3 class="log-topic">${escapeHtml(topic)}</h3>
            ${entries.map((entry) => `
              <div class="log-entry">
                <span class="log-stage">${escapeHtml(cleanText(entry && entry.stage) || "observe")}</span>
                <div class="log-message">${escapeHtml(cleanText(entry && entry.message) || "")}</div>
              </div>
            `).join("")}
          </article>
        `;
      }).join("");
    }

    function toggleAgentLog() {
      if (agentLogSection.hidden) return;
      agentLogOpen = !agentLogOpen;
      agentLogSection.classList.toggle("is-open", agentLogOpen);
      agentLogToggle.setAttribute("aria-expanded", agentLogOpen ? "true" : "false");
    }

    function updateStatBar(data) {
      const articles = filteredArticles(data && data.articles);
      const topics = Array.isArray(data && data.topics) ? data.topics.filter(Boolean) : [];
      const positiveCount = articles.filter((article) => sentimentKey(article.sentiment) === "positive").length;
      statArticles.textContent = articles.length ? String(articles.length) : "—";
      statTopics.textContent = topics.length ? String(topics.length) : "—";
      statPositive.textContent = articles.length ? `${Math.round((positiveCount / articles.length) * 100)}%` : "—";
      statLatest.textContent = articles.length ? relativeTimeFor(articles[0]) : "—";
    }

    async function translateArticles(language, options = {}) {
      const shouldRender = options.renderOnComplete !== false;
      const requestId = ++translationRequestId;
      if (!originalArticles.length) {
        setStatus("Fetch articles before translating the edition.", true);
        return { mode: "skipped" };
      }

      if (language === "English") {
        if (requestId !== translationRequestId) return { mode: "stale" };
        const articles = [...originalArticles];
        if (shouldRender) {
          allArticles = articles;
          renderArticles(allArticles);
          renderTicker(allArticles);
        }
        return { mode: "original", articles };
      }

      try {
        const payload = await postJSON("/translate", {
          articles: originalArticles,
          language
        }, { signal: options.signal });
        if (requestId !== translationRequestId) return { mode: "stale" };
        const articles = Array.isArray(payload.articles) ? payload.articles : [...originalArticles];
        if (shouldRender) {
          allArticles = articles;
          renderArticles(allArticles);
          renderTicker(allArticles);
        }
        return { mode: "translated", articles };
      } catch (error) {
        if (error && (error.code === "ABORTED" || error.name === "AbortError")) return { mode: "aborted" };
        if (requestId !== translationRequestId) return { mode: "stale" };
        const articles = [...originalArticles];
        if (shouldRender) {
          allArticles = articles;
          renderArticles(allArticles);
          renderTicker(allArticles);
        }
        return { mode: "failed", articles };
      }
    }

    async function fetchNews(options = {}) {
      const source = cleanText(options && options.source).toLowerCase() || "desk";
      const topics = topicInputs.map((input) => cleanText(input.value)).filter(Boolean).slice(0, 3);
      if (!topics.length) {
        setStatus("Add at least one topic to build the live edition.", true);
        topicInputs[0].focus();
        return;
      }

      const snapshot = {
        originalArticles: [...originalArticles],
        allArticles: [...allArticles],
        currentTopics: [...currentTopics],
        latestBriefing,
        latestEditionTimestamp,
        reasoningData: JSON.parse(JSON.stringify(reasoningData || {}))
      };
      const requestController = new AbortController();
      let abortHandled = false;
      const restoreAfterAbort = () => {
        if (abortHandled) return;
        abortHandled = true;
        originalArticles = [];
        allArticles = [];
        currentTopics = [];
        latestBriefing = "";
        latestEditionTimestamp = null;
        reasoningData = {};
        renderQuietState();
        renderHeroPreviewCards();
        renderTicker([]);
        renderBriefing("", []);
        renderAgentLog({});
        updateStatBar({ articles: [], topics: [] });
        clearVisualFrames();
        if (resultsContainer) resultsContainer.style.display = "none";
        setStatus("Request cancelled.", false);
        setSynthesisProgress(false, languageSelect.value);
        setFetchLoading(false);
        if (activeFetchController === requestController) activeFetchController = null;
        if (activeFetchReset === restoreAfterAbort) activeFetchReset = null;
      };

      activeFetchController = requestController;
      activeFetchReset = restoreAfterAbort;
      setFetchLoading(true);
      setSynthesisProgress(true, languageSelect.value);
      setStatus("Building the live edition from live sources...", false);
      if (resultsContainer) resultsContainer.style.display = "none";
      renderLoadingState();
      renderBriefing("", []);
      renderAgentLog({});
      clearVisualFrames();
      if (source !== "hero" && editionStage) {
        window.requestAnimationFrame(() => {
          editionStage.scrollIntoView({ behavior: "smooth", block: "start" });
        });
      }

      const selectedLanguage = languageSelect.value;
      const coldStartFetch = !hasSuccessfulFetch;

      try {
        await warmBackend();
        const payload = await fetchJSONWithRetry("/fetch", { topics }, coldStartFetch ? 35000 : 22000, true, coldStartFetch, requestController.signal);
        originalArticles = Array.isArray(payload.articles) ? payload.articles : [];
        currentTopics = Array.isArray(payload.topics) && payload.topics.length ? payload.topics : topics;
        latestBriefing = cleanText(payload.briefing);
        reasoningData = payload.reasoning_log || {};
        lastTopic = currentTopics[0] || lastTopic;
        hasSuccessfulFetch = originalArticles.length > 0 || hasSuccessfulFetch;
        latestEditionTimestamp = new Date();

        if (originalArticles.length) {
          renderHeroEditionCards(originalArticles, currentTopics, {
            createdAt: latestEditionTimestamp
          });
          setSynthesisProgress(false, selectedLanguage);
        }

        let translationResult = { mode: "original" };
        if (selectedLanguage !== "English" && originalArticles.length) {
          translationResult = await translateArticles(selectedLanguage, {
            signal: requestController.signal,
            renderOnComplete: false
          });
        }

        if (!originalArticles.length) {
          setStatus("No live articles came back for those topics.", true);
          return;
        }

        if (translationResult.mode === "aborted") {
          restoreAfterAbort();
          return;
        }

        allArticles = Array.isArray(translationResult.articles) && translationResult.articles.length
          ? [...translationResult.articles]
          : [...originalArticles];

        renderHeroEditionCards(allArticles, currentTopics, {
          animate: false,
          createdAt: latestEditionTimestamp || new Date()
        });
        renderArticles(allArticles);
        renderTicker(allArticles);
        renderBriefing(latestBriefing, currentTopics);
        renderAgentLog(reasoningData);
        updateStatBar({ articles: originalArticles, topics: currentTopics });
        if (originalArticles.length) document.getElementById("results-container").style.display = "block";
        loadVisuals();

        if (translationResult.mode === "translated") {
          setStatus(`Fetched ${originalArticles.length} articles and translated the edition to ${selectedLanguage}.`, false);
        } else if (translationResult.mode === "failed") {
          setStatus(`Fetched ${originalArticles.length} articles, but translation to ${selectedLanguage} failed.`, true);
        } else {
          setStatus(`Fetched ${originalArticles.length} articles across ${currentTopics.length} topic${currentTopics.length === 1 ? "" : "s"}.`, false);
        }
      } catch (error) {
        if (error.code === "ABORTED" || error.name === "AbortError") {
          restoreAfterAbort();
        } else if (error.code === "TIMEOUT") {
          originalArticles = snapshot.originalArticles;
          allArticles = snapshot.allArticles;
          currentTopics = snapshot.currentTopics;
          latestBriefing = snapshot.latestBriefing;
          latestEditionTimestamp = snapshot.latestEditionTimestamp;
          reasoningData = snapshot.reasoningData;
          if (allArticles.length) {
            renderHeroEditionCards(allArticles, currentTopics, {
              animate: false,
              createdAt: latestEditionTimestamp || new Date()
            });
            renderArticles(allArticles);
            renderTicker(allArticles);
            renderBriefing(latestBriefing, currentTopics);
            renderAgentLog(reasoningData);
            updateStatBar({ articles: originalArticles, topics: currentTopics });
            if (resultsContainer) resultsContainer.style.display = "block";
            loadVisuals();
          } else {
            renderDeferredState();
            renderHeroPreviewCards();
            renderTicker([]);
            renderBriefing("", []);
            renderAgentLog({});
            updateStatBar({ articles: [], topics: [] });
            clearVisualFrames();
            if (resultsContainer) resultsContainer.style.display = "none";
          }
          setStatus("The live edition is still compiling. Try a narrower topic mix or retry in a few seconds.", false);
        } else {
          originalArticles = snapshot.originalArticles;
          allArticles = snapshot.allArticles;
          currentTopics = snapshot.currentTopics;
          latestBriefing = snapshot.latestBriefing;
          latestEditionTimestamp = snapshot.latestEditionTimestamp;
          reasoningData = snapshot.reasoningData;
          if (allArticles.length) {
            renderHeroEditionCards(allArticles, currentTopics, {
              animate: false,
              createdAt: latestEditionTimestamp || new Date()
            });
          } else {
            renderHeroPreviewCards();
          }
          renderArticles(allArticles);
          renderTicker(allArticles);
          renderBriefing(latestBriefing, currentTopics);
          renderAgentLog(reasoningData);
          updateStatBar({ articles: originalArticles, topics: currentTopics });
          if (originalArticles.length) {
            if (resultsContainer) resultsContainer.style.display = "block";
            loadVisuals();
          } else {
            renderConnectionRetryState();
            clearVisualFrames();
            if (resultsContainer) resultsContainer.style.display = "none";
          }
          setStatus("Connection Refined - Please Retry", true);
        }
      } finally {
        if (!abortHandled) {
          setSynthesisProgress(false, languageSelect.value);
          setFetchLoading(false);
        }
        if (activeFetchController === requestController) activeFetchController = null;
        if (activeFetchReset === restoreAfterAbort) activeFetchReset = null;
      }
    }

    function toggleChatPanel(forceState) {
      chatPanelOpen = typeof forceState === "boolean" ? forceState : !chatPanelOpen;
      chatPanel.classList.toggle("is-open", chatPanelOpen);
      chatOverlay.classList.toggle("is-visible", chatPanelOpen);
      chatPanel.setAttribute("aria-hidden", chatPanelOpen ? "false" : "true");
      chatFab.setAttribute("aria-expanded", chatPanelOpen ? "true" : "false");
      if (chatPanelOpen) window.setTimeout(() => chatInput.focus(), 100);
    }

    function appendChatBubble(role, content, asHtml) {
      const bubble = document.createElement("div");
      bubble.className = `chat-bubble ${role}`;
      if (asHtml) bubble.innerHTML = content;
      else bubble.textContent = content;
      chatMessages.appendChild(bubble);
      chatMessages.scrollTop = chatMessages.scrollHeight;
    }

    function assistantMarkup(payload) {
      const response = escapeHtml(cleanText(payload && payload.response) || "No response returned.");
      const articles = Array.isArray(payload && payload.articles) ? payload.articles : [];
      const commentary = cleanText(payload && payload.sentiment_commentary);
      const cards = articles.length ? `
        <div class="chat-articles">
          ${articles.map((article) => `
            <div class="chat-article ${sentimentKey(article && article.sentiment)}">
              <p class="chat-article-title">
                <a href="${escapeHtml(safeUrl(article && article.link))}" target="_blank" rel="noopener noreferrer">${escapeHtml(headlineFor(article))}</a>
              </p>
              <div class="chat-article-meta">${escapeHtml(sourceFor(article))} · ${escapeHtml(publishedFor(article))}</div>
            </div>
          `).join("")}
        </div>
      ` : "";
      const commentaryBlock = commentary ? `<div class="chat-commentary">${escapeHtml(commentary)}</div>` : "";
      return `<p>${response}</p>${cards}${commentaryBlock}`;
    }

    function renderSuggestionChips(items) {
      const chips = (items && items.length ? items : DEFAULT_SUGGESTIONS).filter(Boolean).slice(0, 4);
      suggestionChips.innerHTML = chips.map((item) => `
        <button class="chat-chip" type="button" data-chip="${escapeHtml(item)}">${escapeHtml(item)}</button>
      `).join("");
    }

    async function sendChat(message) {
      const trimmed = cleanText(message);
      if (!trimmed) return;

      toggleChatPanel(true);
      appendChatBubble("user", trimmed, false);
      chatHistory.push({ role: "user", content: trimmed });
      chatInput.value = "";
      setChatLoading(true);

      try {
        const payload = await postJSON("/chat", {
          message: trimmed,
          history: chatHistory,
          last_topic: lastTopic,
          articles: originalArticles
        });

        lastTopic = payload.last_topic || lastTopic;
        chatHistory.push({ role: "assistant", content: cleanText(payload.response) });
        appendChatBubble("bot", assistantMarkup(payload), true);
        renderSuggestionChips(payload.suggestions || []);
      } catch (_error) {
        appendChatBubble("bot", "I ran into an issue while checking the latest coverage.", false);
      } finally {
        setChatLoading(false);
      }
    }

    heroFetchBtn.addEventListener("click", () => {
      if (topicInputs.some((input) => cleanText(input.value))) {
        fetchNews({ source: "hero" });
        return;
      }
      deskSection.scrollIntoView({ behavior: "smooth", block: "start" });
      window.setTimeout(() => topicInputs[0].focus(), 280);
    });

    heroInsightsBtn.addEventListener("click", () => {
      insightsBand.scrollIntoView({ behavior: "smooth", block: "start" });
    });

    if (heroBlankStack) {
      heroBlankStack.addEventListener("click", (event) => {
        const trigger = event.target.closest("#heroEditionArrow");
        if (!trigger) return;
        editionStage.scrollIntoView({ behavior: "smooth", block: "start" });
      });
    }

    fetchBtn.addEventListener("click", () => fetchNews({ source: "desk" }));

    if (cancelRequestBtn) {
      cancelRequestBtn.addEventListener("click", () => cancelActiveRequest());
    }

    sampleBtn.addEventListener("click", () => {
      topicInputs[0].value = "Nvidia";
      topicInputs[1].value = "Cricket";
      topicInputs[2].value = "Climate Change";
      fetchNews({ source: "desk" });
    });

    languageSelect.addEventListener("change", async (event) => {
      const result = await translateArticles(event.target.value);
      if (Array.isArray(result.articles) && result.articles.length) {
        renderHeroEditionCards(result.articles, currentTopics, {
          animate: false,
          createdAt: latestEditionTimestamp || new Date()
        });
      }
      if (result.mode === "translated") {
        setStatus(`Showing the edition in ${event.target.value}.`, false);
      } else if (result.mode === "original") {
        setStatus("Showing original English headlines.", false);
      } else if (result.mode === "failed") {
        setStatus(`Translation to ${event.target.value} failed.`, true);
      }
    });

    agentLogToggle.addEventListener("click", () => toggleAgentLog());
    if (themeToggle) {
      themeToggle.addEventListener("click", () => {
        applyTheme(document.body.dataset.theme === "dark" ? "light" : "dark");
      });
    }
    chatFab.addEventListener("click", () => toggleChatPanel());
    chatCloseBtn.addEventListener("click", () => toggleChatPanel(false));
    chatOverlay.addEventListener("click", () => toggleChatPanel(false));
    chatSendBtn.addEventListener("click", () => sendChat(chatInput.value));

    chatInput.addEventListener("keydown", (event) => {
      if (event.key === "Enter") {
        event.preventDefault();
        sendChat(chatInput.value);
      }
    });

    quickBriefingBtn.addEventListener("click", () => sendChat("Give me a briefing"));

    suggestionChips.addEventListener("click", (event) => {
      const chip = event.target.closest("[data-chip]");
      if (!chip) return;
      sendChat(chip.getAttribute("data-chip") || "");
    });

    document.addEventListener("keydown", (event) => {
      if (event.key === "Escape" && chatPanelOpen) toggleChatPanel(false);
    });

    renderHeroPreviewCards();
    renderQuietState();
    renderTicker([]);
    renderBriefing("", []);
    clearVisualFrames();
    renderAgentLog({});
    updateStatBar({ articles: [], topics: [] });
    renderSuggestionChips(DEFAULT_SUGGESTIONS);
    applyTheme(preferredTheme());
    window.requestAnimationFrame(() => {
      window.requestAnimationFrame(() => document.body.classList.add("is-ready"));
    });
    warmBackend();
  </script>
</body>
</html>
"""


## Step 10 — FastAPI Application

This cell writes the FastAPI backend that connects everything together.
FastAPI handles all HTTP requests from the browser and routes them to
the correct Python module.

Five endpoints are exposed:

GET / serves the full HTML page from templates.py

POST /fetch runs the VeritasAgent for up to 3 topics and returns
the processed articles, briefing text, and reasoning log

POST /translate runs the Helsinki-NLP translation pipeline and
returns articles with translated_headline added to each one

POST /chat runs VeritasBot and returns a structured response with
article links and suggestion chips

POST /visuals generates the word cloud and three Plotly charts and
returns them as base64 image data and JSON

CORS middleware is added so the frontend can call the API from the
Cloudflare tunnel domain without cross-origin errors.


In [ ]:
%%writefile /content/main.py
from __future__ import annotations

import threading
from typing import Any

from fastapi import FastAPI
from fastapi.concurrency import run_in_threadpool
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

from agent import load_flan, run_agent_for_topics
from chatbot import generate_chatbot_response
from templates import HTML_PAGE
from translator import translate_articles
from visualizations import build_visual_payload


class FetchRequest(BaseModel):
    topics: list[str] = Field(default_factory=list)


class TranslateRequest(BaseModel):
    articles: list[dict[str, Any]] = Field(default_factory=list)
    language: str = "English"


class ChatRequest(BaseModel):
    message: str
    history: list[dict[str, Any]] = Field(default_factory=list)
    last_topic: str | None = None
    articles: list[dict[str, Any]] = Field(default_factory=list)


class VisualRequest(BaseModel):
    articles: list[dict[str, Any]] = Field(default_factory=list)


app = FastAPI(title="VeritasAI", version="1.0.0")
WARMUP_STATE = {
    "in_progress": False,
    "ready": False,
    "error": None,
}
_warmup_lock = threading.Lock()


def _warm_agent_models() -> None:
    ready = False
    error_message = None
    try:
        tokenizer, model = load_flan()
        ready = tokenizer is not None and model is not None
        if not ready:
            error_message = "Warm-up could not load the FLAN model."
    except Exception as exc:
        error_message = str(exc)
    finally:
        with _warmup_lock:
            WARMUP_STATE["in_progress"] = False
            WARMUP_STATE["ready"] = ready
            WARMUP_STATE["error"] = error_message


def ensure_agent_warmup() -> None:
    with _warmup_lock:
        if WARMUP_STATE["in_progress"] or WARMUP_STATE["ready"]:
            return
        WARMUP_STATE["in_progress"] = True
        WARMUP_STATE["error"] = None
    threading.Thread(target=_warm_agent_models, daemon=True).start()


app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.on_event("startup")
async def startup_event() -> None:
    ensure_agent_warmup()


@app.get("/", response_class=HTMLResponse)
async def index() -> str:
    return HTML_PAGE


@app.get("/health")
async def health() -> dict:
    return {"status": "ok", "warmup": dict(WARMUP_STATE)}


@app.post("/warmup")
async def warmup_endpoint() -> dict:
    ensure_agent_warmup()
    return {"ok": True, "warmup": dict(WARMUP_STATE)}


@app.post("/fetch")
async def fetch_endpoint(request: FetchRequest) -> dict:
    clean_topics = [(topic or "").strip() for topic in request.topics if (topic or "").strip()][:3]
    if not clean_topics:
        return {
            "articles": [],
            "briefing": "",
            "topics": [],
            "reasoning_log": {},
            "topic_briefings": {},
        }
    ensure_agent_warmup()
    return await run_in_threadpool(run_agent_for_topics, clean_topics)


@app.post("/translate")
async def translate_endpoint(request: TranslateRequest) -> dict:
    if (request.language or "English").strip().lower() == "english":
        return {"articles": request.articles}
    translated = await run_in_threadpool(translate_articles, request.articles, request.language)
    return {"articles": translated}


@app.post("/chat")
async def chat_endpoint(request: ChatRequest) -> dict:
    return await run_in_threadpool(
        generate_chatbot_response,
        message=request.message,
        history=request.history,
        last_topic=request.last_topic,
        articles=request.articles,
    )


@app.post("/visuals")
async def visuals_endpoint(request: VisualRequest) -> dict:
    return await run_in_threadpool(build_visual_payload, request.articles)


if __name__ == "__main__":
    import uvicorn

    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=False)


## Step 11 — Syntax Check

This cell runs a quick compile check on every generated Python file
before launch. If any file contains a syntax error this cell will
fail and print exactly which file and which line caused the problem.

This saves time compared to discovering syntax errors after the server
has already started, where the error message is harder to read.

If all files pass the check you will see "All generated modules compiled
successfully" and can proceed to the launch cell.


In [ ]:

import py_compile

module_paths = [
    "/content/fetch_news.py",
    "/content/sentiment.py",
    "/content/nlp_pipeline.py",
    "/content/translator.py",
    "/content/visualizations.py",
    "/content/agent.py",
    "/content/chatbot.py",
    "/content/templates.py",
    "/content/main.py",
]

for module_path in module_paths:
    py_compile.compile(module_path, doraise=True)

print("All generated modules compiled successfully.")


## Step 12 — Launch VeritasAI

This is the final cell. Running it starts the FastAPI server on port 8000
inside the Colab environment, then creates a public HTTPS URL using
Cloudflare Tunnel so the app can be opened from any browser anywhere.

The cell shuts down any previous server or tunnel from earlier in the
session before starting fresh, so it is safe to rerun.

After running this cell:
1. Wait about 15 to 20 seconds for the tunnel to establish
2. A URL ending in trycloudflare.com will print to the output
3. Copy that URL and open it in any browser
4. Enter up to three topics and click Fetch News
5. The agent will run and results will appear in seconds

Important notes:
- The URL is only active while this Colab session is running
- Always use the newest URL printed by this cell; older trycloudflare URLs expire and return DNS errors
- If you ever see DNS_PROBE_FINISHED_NXDOMAIN, rerun only this launch cell and open the new URL it prints
- If Colab disconnects, rerun all cells from the top and run this cell again
- The launch cell now starts warming the FLAN agent before you open the public URL
- The very first fetch can still take longer than later ones if the warm-up is still finishing
- The first translation for each language takes 20 to 40 seconds to download the model
- Every action after those first loads is much faster

Before submitting, set Colab sharing to "Anyone with the link" using
the Share button in the top right corner.


In [ ]:

import json
import os
import re
import subprocess
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

os.chdir("/content")

def tail_text(path: str, lines: int = 20) -> str:
    content = Path(path).read_text(errors="ignore").splitlines()
    return "\n".join(content[-lines:])

for process_name in ("uvicorn_process", "cloudflared_process"):
    process = globals().get(process_name)
    if process and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()

for artifact_path in (
    "/content/uvicorn.log",
    "/content/cloudflared.log",
    "/content/veritas_public_url.txt",
):
    try:
        Path(artifact_path).unlink()
    except FileNotFoundError:
        pass

uvicorn_log = open("/content/uvicorn.log", "w", buffering=1)
cloudflared_log = open("/content/cloudflared.log", "w", buffering=1)

uvicorn_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
)

backend_ready = False
for _ in range(45):
    if uvicorn_process.poll() is not None:
        break
    try:
        with urlopen("http://127.0.0.1:8000/health", timeout=2) as response:
            if getattr(response, "status", 200) == 200:
                backend_ready = True
                break
    except Exception:
        time.sleep(2)

print("FastAPI log file: /content/uvicorn.log")
print("Cloudflare log file: /content/cloudflared.log")

if not backend_ready:
    print("FastAPI did not become healthy. Check /content/uvicorn.log.")
    print(tail_text("/content/uvicorn.log"))
    raise RuntimeError("Backend startup failed before Cloudflare tunnel launch.")

try:
    with urlopen(
        "http://127.0.0.1:8000/warmup",
        data=b"{}",
        timeout=4,
    ) as response:
        warmup_payload = json.loads(response.read().decode("utf-8"))
    warmup_state = warmup_payload.get("warmup", {})
    if warmup_state.get("ready"):
        print("Agent warm-up is already ready.")
    else:
        print("Agent warm-up started in the background.")
except Exception:
    print("Agent warm-up trigger could not be confirmed, but the app will still start.")

cloudflared_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=cloudflared_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    if cloudflared_process.poll() is not None:
        break
    log_text = Path("/content/cloudflared.log").read_text(errors="ignore")
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_text)
    if match:
        candidate_url = match.group(0)
        try:
            with urlopen(f"{candidate_url}/health", timeout=5) as response:
                if getattr(response, "status", 200) == 200:
                    public_url = candidate_url
                    break
        except (HTTPError, URLError, TimeoutError, OSError):
            pass
    time.sleep(2)

if public_url:
    Path("/content/veritas_public_url.txt").write_text(public_url + "\n")
    print("Public frontend URL (use only this newest URL):", public_url)
    print("Saved URL file: /content/veritas_public_url.txt")
else:
    print("Cloudflare did not produce a verified public URL yet.")
    print("If you rerun this cell, only the newest URL remains valid.")
    print("Recent cloudflared log lines:")
    print(tail_text("/content/cloudflared.log"))


## References

All libraries, tools, and external resources used in this project:

| Resource | Purpose | Link |
|---|---|---|
| Google News RSS | Live news data source | https://news.google.com/rss |
| feedparser | RSS feed parsing | https://feedparser.readthedocs.io |
| vaderSentiment | Headline sentiment analysis | https://github.com/cjhutto/vaderSentiment |
| spaCy en_core_web_sm | Named entity recognition | https://spacy.io |
| sumy TextRank | Extractive summarization | https://github.com/miso-belica/sumy |
| Voyant Tools | Visualization inspiration | https://voyant-tools.org |
| wordcloud | Word cloud generation | https://amueller.github.io/word_cloud |
| numpy | Circular mask for word cloud | https://numpy.org |
| Plotly | Interactive charts | https://plotly.com/python |
| python-dateutil | Robust date parsing | https://dateutil.readthedocs.io |
| HuggingFace Transformers | Model loading framework | https://huggingface.co/docs/transformers |
| google/flan-t5-base | Open-source reasoning LLM | https://huggingface.co/google/flan-t5-base |
| Helsinki-NLP OPUS-MT | Open-source translation models | https://huggingface.co/Helsinki-NLP |
| FastAPI | Backend web framework | https://fastapi.tiangolo.com |
| uvicorn | ASGI server | https://www.uvicorn.org |
| Cloudflare Tunnel | Free public HTTPS URL | https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare |
| Inter font | Typography | https://fonts.google.com/specimen/Inter |
| Plotly CDN | Frontend chart rendering | https://cdn.plot.ly/plotly-latest.min.js |
